# Paper 2 — Goal 2 completion, protocol-concordance correction, sensitivities, and final seal

This notebook is run **after Notebook 20 FINAL has completed successfully**. It does not redefine Primary-A or Core-Q. Its purpose is to close Goal 2 completely before Goal 3.

It performs five jobs:

1. verifies the frozen Goal 2 FINAL primary run and its hashes;
2. corrects one protocol-concordance issue in `M_A-resQ`: the frozen Methods require the Q→A ridge penalty to be selected inside the outer-training data by grouped reconstruction CV, whereas Notebook 20 used a fixed alpha=1.0;
3. recomputes the authoritative paired Goal 2 outputs with the corrected residualized representation while leaving `M_A` and `M_A+Q` unchanged;
4. runs the prespecified Goal 2 robustness package: A-only, first-recording diagnosis, median participant aggregation, Extended-Q, robust Q-loss checks, Q-family add-one/leave-one-out, constrained group-safe HGB, overlap diagnostics, and ridge coefficient stability;
5. writes a **Goal 2 final completion seal** and supplementary source tables/figures.

### Important interpretation boundary
Goal 2 remains observational. Adding Q, removing Q-predictable acoustic variation, or finding Q-dependent error demonstrates dependence/sensitivity in this cohort. It does **not** prove a purely technical cause, shortcut learning, or reconstruction of artifact-free physiology.

### Run instruction
Use **Kernel → Restart Kernel and Run All Cells**. This notebook is checkpointed by outer fold so a stopped HGB run can resume safely.


In [1]:
from __future__ import annotations

import hashlib
import importlib.metadata as mdlib
import itertools
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import warnings
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy import stats
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.genmod.cov_struct import Independence, Exchangeable
from statsmodels.genmod.families import Gaussian
from statsmodels.nonparametric.smoothers_lowess import lowess

# ------------------------------------------------------------------
# USER-FACING CONFIG
# ------------------------------------------------------------------

RUN_MODE = "FINAL"    # change to FINAL only after development PASS
RESET_OUTPUTS = False
RESUME_IF_VALID = True

BASE_SEED = 20260825
OUTER_FOLDS = 5
OUTER_REPEATS = 10
INNER_FOLDS = 5

if RUN_MODE not in {"DEVELOPMENT", "FINAL"}:
    raise ValueError("RUN_MODE must be DEVELOPMENT or FINAL.")

N_BOOTSTRAPS = 2000 if RUN_MODE == "FINAL" else 300

# Linear Q->A residualizer. This penalty is prespecified and does not use
# outcome performance. Q and A are standardized inside the training fold.
RESIDUALIZER_ALPHA = 1.0

C_GRID = np.logspace(-4, 4, 9)
ALPHA_GRID = np.logspace(-4, 4, 9)

EXPECTED_PAPER1_COMMIT = (
    "cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8"
)

ENGINE_VERSION = "goal2-primary-v1.1.0"

EXPECTED = {
    "recordings": 519,
    "participants": 224,
    "als_participants": 158,
    "control_participants": 66,
    "als_recordings": 418,
    "control_recordings": 101,
    "diagnosis_age_complete_participants": 199,
    "diagnosis_age_complete_als": 158,
    "diagnosis_age_complete_controls": 41,
    "severity_participants_60d": 145,
    "severity_pairs_60d": 398,
}

def find_project_root() -> Path:
    override = os.environ.get("PAPER2_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if (root / "data" / "processed").exists():
            return root
        raise FileNotFoundError(f"Invalid PAPER2_ROOT: {root}")

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "data" / "processed").exists()
            and (candidate / "data" / "manifests").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate Paper 2 repository root."
    )

ROOT = find_project_root()
PROCESSED = ROOT / "data" / "processed"
MANIFESTS = ROOT / "data" / "manifests"
EXTERNAL = ROOT / "external" / "quality_framework_features"

RUN_TAG = RUN_MODE.lower()
OUT = (
    ROOT
    / "outputs"
    / "goal2"
    / "goal2_primary_v1_1"
    / RUN_TAG
)

TABLES = OUT / "tables"
FIGURES = OUT / "figures"
OOF_DIR = OUT / "oof"
CHECKPOINTS = OUT / "checkpoints"
LOGS = OUT / "logs"

if RESET_OUTPUTS and OUT.exists():
    shutil.rmtree(OUT)

for directory in [
    OUT, TABLES, FIGURES, OOF_DIR, CHECKPOINTS, LOGS
]:
    directory.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

print("Paper 2 root:", ROOT)
print("Engine:", ENGINE_VERSION)
print("Run mode:", RUN_MODE)
print("Bootstrap replicates:", N_BOOTSTRAPS)


Paper 2 root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
Engine: goal2-primary-v1.1.0
Run mode: FINAL
Bootstrap replicates: 2000


## 1. Atomic I/O and reproducibility utilities

In [2]:
def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()

def stable_hash(payload):
    text = json.dumps(
        payload,
        sort_keys=True,
        default=str,
        separators=(",", ":"),
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def atomic_write_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(
        json.dumps(payload, indent=2, default=str),
        encoding="utf-8",
    )
    os.replace(temp, path)

def atomic_write_csv(
    frame,
    path,
    *,
    required_columns=None,
    allow_empty=False,
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{path.name}: expected DataFrame.")

    if not allow_empty and len(frame) == 0:
        raise ValueError(
            f"{path.name}: refusing to write empty table."
        )

    if required_columns:
        missing = set(required_columns) - set(frame.columns)
        if missing:
            raise ValueError(
                f"{path.name}: missing columns {sorted(missing)}"
            )

    temp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temp, index=False)

    if not temp.exists() or temp.stat().st_size == 0:
        raise IOError(
            f"{path.name}: temporary CSV is empty."
        )

    os.replace(temp, path)

def safe_read_csv(
    path,
    *,
    required_columns=None,
    allow_empty=False,
):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    if path.stat().st_size == 0:
        raise IOError(f"{path} exists but is empty.")

    frame = pd.read_csv(path, low_memory=False)

    if not allow_empty and len(frame) == 0:
        raise ValueError(f"{path} has no rows.")

    if required_columns:
        missing = set(required_columns) - set(frame.columns)
        if missing:
            raise ValueError(
                f"{path.name}: missing columns {sorted(missing)}"
            )

    return frame

def package_version(package):
    try:
        return mdlib.version(package)
    except Exception:
        return "unknown"

def deterministic_seed(*tokens, modulus=1_000_000):
    """
    Stable across Python processes and machines.
    Never use Python's built-in hash() for reproducibility.
    """
    digest = stable_hash(list(tokens))
    return BASE_SEED + (int(digest[:16], 16) % modulus)

def coerce_bool_series(series, name):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    normalized = (
        series.astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "y": True,
        "false": False,
        "0": False,
        "no": False,
        "n": False,
    }

    unknown = sorted(
        set(normalized.dropna()) - set(mapping)
    )
    if unknown:
        raise ValueError(
            f"{name}: unrecognized boolean values: {unknown[:20]}"
        )

    return normalized.map(mapping).astype(bool)

print("Atomic I/O / deterministic utilities: READY")


Atomic I/O / deterministic utilities: READY


## 2. Hard A/Q/cohort/split gates

The notebook will not fit a model unless:

- the A freeze manifest says `FROZEN`;
- the A values/registry hashes match;
- the canonical 519/224/158/66 denominators still hold;
- the Goal 1 split manifest is intact;
- the Phase 0.5 QCHAN cache still reproduces the frozen Paper 1 QCHAN release.


In [3]:
PATHS = {
    "recording_table": PROCESSED / "recording_table_phase0.csv",
    "participant_table": PROCESSED / "participant_table.csv",
    "severity_pairs": PROCESSED / "severity_pairs.csv",
    "a_values": PROCESSED / "acoustic_features_frozen.csv",
    "a_registry": MANIFESTS / "a_registry.csv",
    "a_manifest": MANIFESTS / "a_freeze_manifest.json",
    "q_registry": MANIFESTS / "q_registry.csv",
    "split_manifest": MANIFESTS / "split_manifest.csv",
    "qchan_ready": MANIFESTS / "qchan_cache_ready.json",
    "qchan_cache_index": MANIFESTS / "qchan_spectrum_cache_index.csv",
}

missing = [
    name for name, path in PATHS.items()
    if not path.exists()
]
if missing:
    raise FileNotFoundError(
        "Missing Goal 2 prerequisite(s): "
        + ", ".join(missing)
    )

recording_table = safe_read_csv(
    PATHS["recording_table"],
    required_columns=[
        "participant_id",
        "logical_recording_id",
        "diagnosis",
        "age_at_recording_years",
    ],
)

participant_table = safe_read_csv(
    PATHS["participant_table"],
    required_columns=[
        "participant_id",
        "diagnosis",
    ],
)

severity_pairs_raw = safe_read_csv(
    PATHS["severity_pairs"],
    required_columns=[
        "participant_id",
        "logical_recording_id",
        "bulbar_score",
        "abs_delta_days",
        "within_60_days",
    ],
)

a_values = safe_read_csv(
    PATHS["a_values"],
    required_columns=[
        "participant_id",
        "logical_recording_id",
    ],
)

a_registry = safe_read_csv(
    PATHS["a_registry"],
    required_columns=[
        "feature",
        "final_role",
        "final_transform",
        "support_indicator_column",
        "a_freeze_version",
    ],
)

q_registry = safe_read_csv(
    PATHS["q_registry"],
    required_columns=[
        "feature",
        "family",
        "paper2_role",
        "transform",
    ],
)

split_manifest = safe_read_csv(
    PATHS["split_manifest"],
    required_columns=[
        "participant_id",
        "repeat",
        "outer_fold",
    ],
)

qchan_cache_index = safe_read_csv(
    PATHS["qchan_cache_index"],
    required_columns=[
        "logical_recording_id",
        "status",
        "spectrum_sha256",
        "cache_path",
    ],
)

a_manifest = json.loads(
    PATHS["a_manifest"].read_text(encoding="utf-8")
)
qchan_ready = json.loads(
    PATHS["qchan_ready"].read_text(encoding="utf-8")
)

for frame in [
    recording_table,
    participant_table,
    severity_pairs_raw,
    a_values,
    split_manifest,
    qchan_cache_index,
]:
    if "participant_id" in frame.columns:
        frame["participant_id"] = (
            frame["participant_id"]
            .astype(str)
            .str.strip()
        )
    if "logical_recording_id" in frame.columns:
        frame["logical_recording_id"] = (
            frame["logical_recording_id"]
            .astype(str)
            .str.strip()
        )

if a_manifest.get("status") != "FROZEN":
    raise RuntimeError(
        "A freeze manifest is not FROZEN."
    )

if sha256_file(PATHS["a_registry"]) != a_manifest["a_registry_sha256"]:
    raise RuntimeError("Frozen A registry hash mismatch.")

if (
    sha256_file(PATHS["a_values"])
    != a_manifest["acoustic_features_frozen_sha256"]
):
    raise RuntimeError("Frozen A values hash mismatch.")

if a_manifest["n_primary_a"] != 6:
    raise RuntimeError(
        "Expected six frozen Primary-A features."
    )

# Canonical denominators.
recording_table["y_dx"] = recording_table[
    "diagnosis"
].map({"ALS": 1, "CONTROLS": 0})

if recording_table["y_dx"].isna().any():
    raise ValueError("Unexpected diagnosis label.")

assert len(recording_table) == EXPECTED["recordings"]
assert (
    recording_table["participant_id"].nunique()
    == EXPECTED["participants"]
)
assert int((recording_table["y_dx"] == 1).sum()) == EXPECTED["als_recordings"]
assert int((recording_table["y_dx"] == 0).sum()) == EXPECTED["control_recordings"]

participant_dx = (
    recording_table[
        ["participant_id", "y_dx"]
    ]
    .drop_duplicates()
)

assert len(participant_dx) == EXPECTED["participants"]
assert int(participant_dx["y_dx"].sum()) == EXPECTED["als_participants"]
assert int((participant_dx["y_dx"] == 0).sum()) == EXPECTED["control_participants"]

# A identity alignment.
if len(a_values) != EXPECTED["recordings"]:
    raise RuntimeError(
        f"Frozen A table has {len(a_values)} rows, expected 519."
    )
if not a_values["logical_recording_id"].is_unique:
    raise RuntimeError("Frozen A recording IDs are not unique.")
if set(a_values["logical_recording_id"]) != set(
    recording_table["logical_recording_id"]
):
    raise RuntimeError(
        "Frozen A identities differ from canonical recordings."
    )

# Split contract.
assert len(split_manifest) == EXPECTED["participants"] * OUTER_REPEATS
assert split_manifest["repeat"].nunique() == OUTER_REPEATS
assert split_manifest["outer_fold"].nunique() == OUTER_FOLDS
assert split_manifest.groupby(
    ["participant_id", "repeat"]
).size().eq(1).all()
assert set(split_manifest["participant_id"]) == set(
    participant_dx["participant_id"]
)

# QCHAN readiness.
for flag in [
    "all_media_paths_resolved",
    "all_media_hashes_verified",
    "all_spectra_measured",
    "paper1_release_reproduced",
]:
    if qchan_ready.get(flag) is not True:
        raise RuntimeError(
            f"QCHAN readiness failure: {flag}"
        )

if (
    qchan_ready.get("paper1_repo_commit")
    != EXPECTED_PAPER1_COMMIT
):
    raise RuntimeError(
        "QCHAN cache was not generated from pinned Paper 1 commit."
    )

if not (EXTERNAL / ".git").exists():
    raise FileNotFoundError(
        f"Pinned Paper 1 repo missing: {EXTERNAL}"
    )

observed_paper1_commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(EXTERNAL),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

if observed_paper1_commit != EXPECTED_PAPER1_COMMIT:
    raise RuntimeError(
        "Pinned Paper 1 checkout changed."
    )

print("A / COHORT / SPLIT / QCHAN GATES: PASS")
print(
    "519 recordings | 224 participants | "
    "158 ALS | 66 controls"
)
print(
    "Frozen A:",
    a_manifest["n_primary_a"],
    "Primary |",
    a_manifest["n_extended_a"],
    "Extended",
)


A / COHORT / SPLIT / QCHAN GATES: PASS
519 recordings | 224 participants | 158 ALS | 66 controls
Frozen A: 6 Primary | 22 Extended


## 3. Freeze the exact Goal 2 model representations

Primary-A is read from `a_registry.csv`; no feature identity is hard-coded from observed results.

Core-Q is the exact Goal 1 representation:

- QADD 3 numeric + 3 support indicators;
- QGAIN 4;
- QREV normalized-fast SRMR;
- QCHAN 4 fold-safe reference-relative values.


In [4]:
AGE = "age_at_recording_years"

PRIMARY_A = a_registry.loc[
    a_registry["final_role"].eq("primary"),
    "feature",
].astype(str).tolist()

EXTENDED_A = a_registry.loc[
    a_registry["final_role"].eq("extended"),
    "feature",
].astype(str).tolist()

A_TRANSFORM = (
    a_registry
    .set_index("feature")["final_transform"]
    .astype(str)
    .to_dict()
)

A_SUPPORT = (
    a_registry
    .set_index("feature")["support_indicator_column"]
    .astype(str)
    .to_dict()
)

QADD = [
    "qadd_pause_ac_level_dbfs_median",
    "qadd_pause_level_iqr_db",
    "qadd_speech_pause_level_contrast_db",
]

QGAIN = [
    "qgain_typical_speech_level_dbfs",
    "qgain_within_segment_iqr_db",
    "qgain_between_segment_mad_db",
    "qgain_abs_drift_db_per_min",
]

QREV = [
    "qrev_srmr_norm",
]

# QCHAN imported below from the validated Paper 1 implementation.
QCHAN = []

Q_TRANSFORM = dict(
    zip(
        q_registry["feature"].astype(str),
        q_registry["transform"].astype(str),
    )
)

Q_SUPPORT = [
    f"{feature}_supported"
    for feature in QADD
]

# Verify all frozen A columns exist.
for feature in PRIMARY_A + EXTENDED_A:
    if feature not in a_values.columns:
        raise KeyError(
            f"Frozen A feature missing from values: {feature}"
        )

for feature in PRIMARY_A:
    support_col = A_SUPPORT[feature]
    if support_col not in a_values.columns:
        raise KeyError(
            f"Primary-A support indicator missing: {support_col}"
        )

print("Primary-A:")
display(
    a_registry.loc[
        a_registry["final_role"].eq("primary"),
        [
            "feature",
            "acoustic_family",
            "speech_subsystem",
            "final_transform",
            "support_indicator_column",
        ],
    ]
)

print("MODEL REPRESENTATION CONTRACT: PART 1 PASS")


Primary-A:


,feature,acoustic_family,speech_subsystem,final_transform,support_indicator_column
5,bamboo_percent_pause_time_300ms,timing_fluency,respiratory_prosodic_temporal,log1p,bamboo_percent_pause_time_300ms__supported
7,bamboo_pause_mean_sec_300ms,timing_fluency,respiratory_prosodic_temporal,log1p,bamboo_pause_mean_sec_300ms__supported
12,bamboo_phrase_mean_sec_300ms,timing_fluency,respiratory_prosodic_temporal,log1p,bamboo_phrase_mean_sec_300ms__supported
15,bamboo_phrase_cv_300ms,timing_fluency,respiratory_prosodic_temporal,log1p,bamboo_phrase_cv_300ms__supported
18,bamboo_nominal_articulation_rate_syll_per_sec,timing_fluency,articulatory_temporal,none,bamboo_nominal_articulation_rate_syll_per_sec_...
22,bamboo_f0_iqr_semitones,phonation_prosody,phonatory_prosodic,log1p,bamboo_f0_iqr_semitones__supported


MODEL REPRESENTATION CONTRACT: PART 1 PASS


## 4. Load the validated reference-independent QCHAN cache

Every inner/outer predictive QCHAN value is reconstructed from the relevant training participants only.


In [5]:
paper1_src = EXTERNAL / "src"
if str(paper1_src) not in sys.path:
    sys.path.insert(0, str(paper1_src))

from paper1_qc_reviewed.qchan_v400 import (
    ANALYSIS_FEATURES as QCHAN_FEATURES_TUPLE,
    DEFAULT_PARAMETERS as QCHAN_PARAMETERS,
    build_subject_balanced_loso_references,
    compute_reference_relative_features,
)
from paper1_qc_reviewed.qchan_v400_cohort import (
    load_recording_spectrum,
)

QCHAN = list(QCHAN_FEATURES_TUPLE)
CORE_Q = QADD + QGAIN + QREV + QCHAN

for feature in CORE_Q:
    if feature not in Q_TRANSFORM:
        raise KeyError(
            f"Core-Q transform missing from q_registry: {feature}"
        )

for support in Q_SUPPORT:
    if support not in recording_table.columns:
        raise KeyError(
            f"Core-Q support indicator missing: {support}"
        )

spectra = {}

for row in qchan_cache_index.itertuples(index=False):
    cache_path = Path(str(row.cache_path))
    if not cache_path.is_absolute():
        cache_path = ROOT / cache_path

    if not cache_path.exists():
        raise FileNotFoundError(
            f"QCHAN cache missing: {cache_path}"
        )

    spectrum = load_recording_spectrum(
        cache_path
    )

    if spectrum.status != "measured":
        raise RuntimeError(
            f"QCHAN spectrum not measured: "
            f"{row.logical_recording_id}"
        )

    spectra[str(row.logical_recording_id)] = spectrum

if len(spectra) != EXPECTED["recordings"]:
    raise RuntimeError(
        f"Expected 519 QCHAN spectra, got {len(spectra)}."
    )

QCHAN_CACHE = OrderedDict()
QCHAN_CACHE_LIMIT = 2048

def qchan_cache_key(reference_participants, target_rows):
    targets = (
        target_rows[
            ["participant_id", "logical_recording_id"]
        ]
        .drop_duplicates()
        .sort_values(
            ["participant_id", "logical_recording_id"]
        )
    )

    payload = {
        "reference_participants": sorted(
            map(str, reference_participants)
        ),
        "targets": (
            targets.astype(str)
            .to_dict("records")
        ),
        "paper1_commit": observed_paper1_commit,
    }

    return stable_hash(payload)

def reference_metadata(reference_rows):
    meta = (
        reference_rows[
            ["logical_recording_id", "participant_id"]
        ]
        .drop_duplicates()
        .rename(
            columns={"participant_id": "subject_id"}
        )
        .copy()
    )

    meta["logical_recording_id"] = (
        meta["logical_recording_id"]
        .astype(str)
    )
    meta["subject_id"] = (
        meta["subject_id"]
        .astype(str)
    )
    meta["task_stratum"] = "BAMBOO_PASSAGE"
    return meta

def fold_safe_qchan(
    reference_participant_ids,
    target_rows,
):
    reference_participant_ids = set(
        map(str, reference_participant_ids)
    )

    if not reference_participant_ids:
        raise ValueError(
            "QCHAN reference participant set is empty."
        )

    target_unique = (
        target_rows[
            ["participant_id", "logical_recording_id"]
        ]
        .drop_duplicates()
        .copy()
    )

    target_unique["participant_id"] = (
        target_unique["participant_id"].astype(str)
    )
    target_unique["logical_recording_id"] = (
        target_unique[
            "logical_recording_id"
        ].astype(str)
    )

    if target_unique[
        "logical_recording_id"
    ].duplicated().any():
        raise ValueError(
            "A recording ID maps to multiple participants "
            "in QCHAN target rows."
        )

    key = qchan_cache_key(
        reference_participant_ids,
        target_unique,
    )

    if key in QCHAN_CACHE:
        cached = QCHAN_CACHE.pop(key)
        QCHAN_CACHE[key] = cached
        return cached.copy()

    # Important: QCHAN's measurement reference uses ALL retained
    # canonical recordings belonging to the training participants,
    # participant-balanced and outcome-blind.
    reference_rows = recording_table.loc[
        recording_table["participant_id"].isin(
            reference_participant_ids
        ),
        [
            "participant_id",
            "logical_recording_id",
        ],
    ].drop_duplicates()

    observed_reference_participants = set(
        reference_rows[
            "participant_id"
        ].astype(str)
    )

    if (
        observed_reference_participants
        != reference_participant_ids
    ):
        missing = (
            reference_participant_ids
            - observed_reference_participants
        )
        raise ValueError(
            "QCHAN training participants missing canonical "
            f"recordings: {sorted(missing)[:10]}"
        )

    ref_spectra = {
        rid: spectra[rid]
        for rid in reference_rows[
            "logical_recording_id"
        ].astype(str)
    }

    ref_meta = reference_metadata(
        reference_rows
    )

    training_references = (
        build_subject_balanced_loso_references(
            ref_spectra,
            ref_meta,
            parameters=QCHAN_PARAMETERS,
        )
    )

    external = target_unique.loc[
        ~target_unique[
            "participant_id"
        ].isin(reference_participant_ids)
    ]

    external_reference = None

    if len(external):
        dummy = external.iloc[0]
        dummy_rid = str(
            dummy["logical_recording_id"]
        )
        dummy_pid = str(
            dummy["participant_id"]
        )

        combo_spectra = dict(ref_spectra)
        combo_spectra[dummy_rid] = spectra[dummy_rid]

        combo_meta = pd.concat(
            [
                ref_meta,
                pd.DataFrame(
                    [{
                        "logical_recording_id": dummy_rid,
                        "subject_id": dummy_pid,
                        "task_stratum": "BAMBOO_PASSAGE",
                    }]
                ),
            ],
            ignore_index=True,
        )

        refs = (
            build_subject_balanced_loso_references(
                combo_spectra,
                combo_meta,
                parameters=QCHAN_PARAMETERS,
            )
        )

        external_reference = refs[dummy_rid]

        members = set(
            map(
                str,
                external_reference.member_subject_ids,
            )
        )

        if not members.issubset(
            reference_participant_ids
        ):
            raise RuntimeError(
                "HELD-OUT QCHAN REFERENCE LEAKAGE."
            )

        if dummy_pid in members:
            raise RuntimeError(
                "External dummy participant entered "
                "its own QCHAN reference."
            )

    output_rows = []

    for row in target_unique.itertuples(
        index=False
    ):
        pid = str(row.participant_id)
        rid = str(row.logical_recording_id)

        if pid in reference_participant_ids:
            reference = training_references[rid]
            members = set(
                map(
                    str,
                    reference.member_subject_ids,
                )
            )

            if pid in members:
                raise RuntimeError(
                    f"QCHAN LOSO failure: {pid}"
                )

            if not members.issubset(
                reference_participant_ids
            ):
                raise RuntimeError(
                    "QCHAN training reference leakage."
                )

        else:
            if external_reference is None:
                raise RuntimeError(
                    "Missing external QCHAN reference."
                )
            reference = external_reference

        values = compute_reference_relative_features(
            spectra[rid],
            reference,
            parameters=QCHAN_PARAMETERS,
        )

        output_rows.append({
            "participant_id": pid,
            "logical_recording_id": rid,
            **{
                feature: values[feature]
                for feature in QCHAN
            },
        })

    result = pd.DataFrame(
        output_rows
    )

    if len(result) != len(target_unique):
        raise RuntimeError(
            "QCHAN row-count mismatch."
        )

    if result[QCHAN].isna().any().any():
        raise RuntimeError(
            "Unexpected missing fold-safe QCHAN."
        )

    QCHAN_CACHE[key] = result.copy()

    while len(QCHAN_CACHE) > QCHAN_CACHE_LIMIT:
        QCHAN_CACHE.popitem(last=False)

    return result

print(
    "QCHAN CACHE + FOLD-SAFE RECONSTRUCTION: READY"
)


QCHAN CACHE + FOLD-SAFE RECONSTRUCTION: READY


## 5. Frozen run signature

Checkpoint reuse is allowed only under the identical A/Q hashes, cohort inputs, seeds, folds, model grids, and residualizer contract. A stale OOF file from another analysis contract cannot be silently reused.


In [6]:
input_hashes_for_signature = {
    name: sha256_file(path)
    for name, path in PATHS.items()
    if Path(path).is_file()
}

run_contract = {
    "engine_version": ENGINE_VERSION,
    "run_mode": RUN_MODE,
    "base_seed": BASE_SEED,
    "outer_folds": OUTER_FOLDS,
    "outer_repeats": OUTER_REPEATS,
    "inner_folds": INNER_FOLDS,
    "bootstrap_replicates": N_BOOTSTRAPS,
    "residualizer_alpha": RESIDUALIZER_ALPHA,
    "C_grid": C_GRID.tolist(),
    "alpha_grid": ALPHA_GRID.tolist(),
    "primary_A": PRIMARY_A,
    "primary_A_transforms": {
        f: A_TRANSFORM[f] for f in PRIMARY_A
    },
    "core_Q": CORE_Q,
    "core_Q_transforms": {
        f: Q_TRANSFORM[f] for f in CORE_Q
    },
    "Q_support": Q_SUPPORT,
    "paper1_commit": observed_paper1_commit,
    "input_hashes": input_hashes_for_signature,
}

RUN_SIGNATURE = stable_hash(run_contract)
signature_path = OUT / "run_signature.json"

if signature_path.exists() and not RESET_OUTPUTS:
    existing = json.loads(
        signature_path.read_text(encoding="utf-8")
    )
    if existing.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing Goal 2 output directory belongs to a different "
            "run contract. Set RESET_OUTPUTS=True once to deliberately "
            "clear it, or use the matching notebook/configuration."
        )

atomic_write_json(
    {
        "run_signature": RUN_SIGNATURE,
        **run_contract,
    },
    signature_path,
)

print("RUN SIGNATURE:", RUN_SIGNATURE[:16])
print("RUN-SIGNATURE GATE: PASS")


RUN SIGNATURE: 5f15eb4ba85fff79
RUN-SIGNATURE GATE: PASS


## 6. Construct exact Goal 2 analysis populations

Primary diagnosis uses all retained recordings from participants with observed age, because age is included unchanged in all three primary representations and age is **not imputed**.

Primary severity uses all frozen ≤60-day recording–assessment pairs. Participant remains the split/weighting cluster.


In [7]:
# Join frozen A to canonical recording rows.
recording_model = recording_table.merge(
    a_values,
    on=[
        "participant_id",
        "logical_recording_id",
    ],
    how="left",
    validate="one_to_one",
    suffixes=("", "_A"),
    indicator="_a_join",
)

if not recording_model[
    "_a_join"
].eq("both").all():
    raise RuntimeError(
        "At least one canonical recording lacks frozen A."
    )

recording_model = recording_model.drop(
    columns=["_a_join"]
)

# Diagnosis primary: age-complete participants only.
dx = recording_model.loc[
    recording_model[AGE].notna()
].copy()

dx["y"] = dx["diagnosis"].map({
    "ALS": 1,
    "CONTROLS": 0,
})

if dx["y"].isna().any():
    raise ValueError(
        "Unexpected diagnosis coding."
    )

dx_participants = (
    dx[
        ["participant_id", "y"]
    ]
    .drop_duplicates()
)

assert (
    dx_participants["participant_id"].nunique()
    == EXPECTED[
        "diagnosis_age_complete_participants"
    ]
)

assert int(dx_participants["y"].sum()) == EXPECTED[
    "diagnosis_age_complete_als"
]

assert int(
    (dx_participants["y"] == 0).sum()
) == EXPECTED[
    "diagnosis_age_complete_controls"
]

# ------------------------------------------------------------------
# Severity primary: compact frozen pair table + canonical Q/demographic
# rows + frozen A.
# ------------------------------------------------------------------

severity_pairs_raw["within_60_days"] = coerce_bool_series(
    severity_pairs_raw["within_60_days"],
    "within_60_days",
)

severity_pairs_60 = severity_pairs_raw.loc[
    severity_pairs_raw["within_60_days"]
].copy()

pair_fields = [
    "participant_id",
    "logical_recording_id",
    "bulbar_score",
    "abs_delta_days",
]

for optional in [
    "recording_date",
    "assessment_date",
    "alsfrs_total",
    "delta_days",
]:
    if optional in severity_pairs_60.columns:
        pair_fields.append(optional)

pair_meta = severity_pairs_60[
    pair_fields
].copy()

clinical_collision = {
    "bulbar_score",
    "alsfrs_total",
    "assessment_date",
    "delta_days",
    "abs_delta_days",
    "within_60_days",
    "within_90_days",
}

canonical_for_pair = recording_model.drop(
    columns=[
        c for c in clinical_collision
        if c in recording_model.columns
    ],
    errors="ignore",
).copy()

if (
    "recording_date" in pair_meta.columns
    and "recording_date" in canonical_for_pair.columns
):
    canonical_for_pair = canonical_for_pair.drop(
        columns=["recording_date"]
    )

sev = pair_meta.merge(
    canonical_for_pair,
    on=[
        "participant_id",
        "logical_recording_id",
    ],
    how="left",
    validate="many_to_one",
    indicator="_recording_join",
)

if len(sev) != len(pair_meta):
    raise RuntimeError(
        "Severity enrichment changed pair-row count."
    )

if not sev[
    "_recording_join"
].eq("both").all():
    raise RuntimeError(
        "Severity pair failed canonical+A join."
    )

sev = sev.drop(
    columns=["_recording_join"]
)

sev["y"] = pd.to_numeric(
    sev["bulbar_score"],
    errors="raise",
)

if not sev["y"].between(0, 12).all():
    raise ValueError(
        "Bulbar score outside 0–12."
    )

if not sev["abs_delta_days"].le(60).all():
    raise RuntimeError(
        "Primary severity includes >60-day pair."
    )

if sev[AGE].isna().any():
    raise RuntimeError(
        "Primary severity contains missing age."
    )

assert len(sev) == EXPECTED["severity_pairs_60d"]
assert (
    sev["participant_id"].nunique()
    == EXPECTED["severity_participants_60d"]
)

# Participant-normalized row weights.
def participant_weights(frame):
    counts = frame.groupby(
        "participant_id"
    )["participant_id"].transform("size")

    weights = 1.0 / counts.to_numpy(float)

    # Every participant must sum to exactly one total fitting weight.
    check = (
        pd.DataFrame({
            "participant_id": frame["participant_id"].to_numpy(),
            "weight": weights,
        })
        .groupby("participant_id")["weight"]
        .sum()
    )

    if not np.allclose(
        check.to_numpy(float),
        1.0,
        atol=1e-12,
        rtol=0,
    ):
        raise RuntimeError(
            "Participant weights do not sum to 1."
        )

    return weights

dx["row_weight"] = participant_weights(dx)
sev["row_weight"] = participant_weights(sev)

# Every analysis participant must exist in the master split.
if not set(dx["participant_id"]).issubset(
    set(split_manifest["participant_id"])
):
    raise RuntimeError(
        "Diagnosis participant absent from split manifest."
    )

if not set(sev["participant_id"]).issubset(
    set(split_manifest["participant_id"])
):
    raise RuntimeError(
        "Severity participant absent from split manifest."
    )

population_summary = pd.DataFrame([
    {
        "branch": "diagnosis",
        "rows": len(dx),
        "participants": dx["participant_id"].nunique(),
        "ALS_participants": int(
            dx_participants["y"].sum()
        ),
        "control_participants": int(
            (dx_participants["y"] == 0).sum()
        ),
    },
    {
        "branch": "severity_60d",
        "rows": len(sev),
        "participants": sev["participant_id"].nunique(),
        "ALS_participants": sev["participant_id"].nunique(),
        "control_participants": 0,
    },
])

atomic_write_csv(
    population_summary,
    TABLES / "goal2_population_summary.csv",
)

display(population_summary)
print("PRIMARY ANALYSIS POPULATIONS: PASS")


,branch,rows,participants,ALS_participants,control_participants
0,diagnosis,483,199,158,41
1,severity_60d,398,145,145,0


PRIMARY ANALYSIS POPULATIONS: PASS


## 7. Frozen transformations and training-only preprocessors

- A transformations come only from the frozen A registry.
- Q transformations come only from the frozen Q registry.
- Median imputation and centering/scaling are learned from training rows only.
- A and Q support indicators remain explicit predictors.


In [8]:
def apply_transform(values, transform):
    x = pd.to_numeric(
        values,
        errors="coerce",
    ).to_numpy(float)

    transform = str(transform).strip().lower()

    if transform in {"none", "", "nan"}:
        return x

    if transform == "log1p":
        finite = np.isfinite(x)
        if finite.any() and np.nanmin(x[finite]) < 0:
            raise ValueError(
                "log1p requested for negative values."
            )
        return np.log1p(x)

    if transform == "asinh":
        return np.arcsinh(x)

    raise ValueError(
        f"Unsupported frozen transform: {transform!r}"
    )

class NumericPreprocessor:
    def __init__(self, features, transform_map):
        self.features = list(features)
        self.transform_map = dict(transform_map)
        self.medians = None
        self.scaler = None

    def _raw(self, frame):
        columns = []

        for feature in self.features:
            if feature not in frame.columns:
                raise KeyError(
                    f"Missing numeric predictor: {feature}"
                )

            columns.append(
                apply_transform(
                    frame[feature],
                    self.transform_map[feature],
                )
            )

        if not columns:
            return np.empty(
                (len(frame), 0),
                dtype=float,
            )

        return np.column_stack(columns)

    def fit(self, frame):
        X = self._raw(frame)

        medians = np.nanmedian(
            X,
            axis=0,
        )

        if np.isnan(medians).any():
            bad = [
                feature
                for feature, median
                in zip(self.features, medians)
                if np.isnan(median)
            ]
            raise ValueError(
                "Training fold has no finite support for: "
                + ", ".join(bad)
            )

        self.medians = medians

        X_imp = np.where(
            np.isnan(X),
            medians[None, :],
            X,
        )

        self.scaler = StandardScaler()
        self.scaler.fit(X_imp)

        return self

    def transform(self, frame):
        if self.medians is None or self.scaler is None:
            raise RuntimeError(
                "NumericPreprocessor is not fitted."
            )

        X = self._raw(frame)
        X_imp = np.where(
            np.isnan(X),
            self.medians[None, :],
            X,
        )

        return self.scaler.transform(
            X_imp
        )

def get_binary_matrix(frame, columns):
    if not columns:
        return np.empty(
            (len(frame), 0),
            dtype=float,
        )

    missing = [
        c for c in columns
        if c not in frame.columns
    ]
    if missing:
        raise KeyError(
            "Missing support predictor(s): "
            + ", ".join(missing)
        )

    X = frame[columns].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(float)

    if not np.isfinite(X).all():
        raise ValueError(
            "Support indicator contains missing/nonfinite values."
        )

    if not set(np.unique(X)).issubset({0.0, 1.0}):
        raise ValueError(
            "Support indicator is not binary."
        )

    return X

class FinalDesignScaler:
    def __init__(self):
        self.scaler = StandardScaler()

    def fit_transform(self, X):
        return self.scaler.fit_transform(X)

    def transform(self, X):
        return self.scaler.transform(X)

print("PREPROCESSING UTILITIES: READY")


PREPROCESSING UTILITIES: READY


## 8. Build fold-specific A/Q and the three paired representations

The same training/validation rows are used for all three models.

`A_resQ` removes only the **linear** Q-predictable component under the prespecified ridge map. It does not claim complete removal of Q information.


In [9]:
def attach_fold_safe_qchan(
    frame,
    reference_ids,
):
    frame = frame.copy()

    qchan = fold_safe_qchan(
        reference_ids,
        frame,
    )

    frame = frame.drop(
        columns=[
            c for c in QCHAN
            if c in frame.columns
        ],
        errors="ignore",
    )

    merged = frame.merge(
        qchan,
        on=[
            "participant_id",
            "logical_recording_id",
        ],
        how="left",
        validate="many_to_one",
    )

    if merged[QCHAN].isna().any().any():
        raise RuntimeError(
            "Fold-safe QCHAN merge produced missing values."
        )

    return merged


class PerFeatureQResidualizer:
    """
    Fit one ridge Q->A map per acoustic feature.

    Critical support rule:
    each A_j map is fit only on training rows where A_j is observed/supported.
    Unsupported A rows are not used as pseudo-observed median targets.

    QADD support indicators are included in the Q design because they are
    part of Core-Q. A support indicators are NOT residualized.
    """

    def __init__(self, alpha):
        self.alpha = float(alpha)
        self.models = []
        self.q_scaler = StandardScaler()

    def fit(
        self,
        Q_numeric,
        Q_support,
        A_numeric,
        A_support,
        sample_weight,
    ):
        q_raw = np.column_stack([
            Q_numeric,
            Q_support,
        ])

        Q_design = self.q_scaler.fit_transform(
            q_raw
        )

        self.models = []

        for j in range(A_numeric.shape[1]):
            supported = (
                A_support[:, j] == 1
            )

            if supported.sum() < 20:
                raise RuntimeError(
                    "Too few supported training rows to fit "
                    f"Q->A residualizer for Primary-A column {j}: "
                    f"{supported.sum()}"
                )

            model = Ridge(
                alpha=self.alpha,
                fit_intercept=True,
            )

            model.fit(
                Q_design[supported],
                A_numeric[supported, j],
                sample_weight=sample_weight[supported],
            )

            self.models.append(model)

        return self

    def transform(
        self,
        Q_numeric,
        Q_support,
        A_numeric,
        A_support,
    ):
        if not self.models:
            raise RuntimeError(
                "PerFeatureQResidualizer is not fitted."
            )

        q_raw = np.column_stack([
            Q_numeric,
            Q_support,
        ])

        Q_design = self.q_scaler.transform(
            q_raw
        )

        residual = np.asarray(
            A_numeric,
            dtype=float,
        ).copy()

        for j, model in enumerate(self.models):
            supported = (
                A_support[:, j] == 1
            )

            predicted = model.predict(
                Q_design
            )

            # Only an actually observed A value is residualized.
            residual[supported, j] = (
                A_numeric[supported, j]
                - predicted[supported]
            )

            # If A is unavailable, preserve the training-fold imputed
            # numeric placeholder and its separate support indicator.
            # Do not manufacture a Q-derived residual for an unobserved A.
            residual[~supported, j] = (
                A_numeric[~supported, j]
            )

        return residual


def fit_fold_representation(
    train,
    target,
    *,
    reference_ids,
    a_features,
    q_features,
):
    """
    Fit every data-dependent operation on train only.

    Returns the same paired rows for:
      M_A      = Age + A + A_support
      M_A+Q    = Age + A + A_support + Q + Q_support
      M_A-resQ = Age + A_resQ + A_support
    """
    reference_ids = set(
        map(str, reference_ids)
    )

    train_q = attach_fold_safe_qchan(
        train,
        reference_ids,
    )

    target_q = attach_fold_safe_qchan(
        target,
        reference_ids,
    )

    # ----------------------------------------------------------
    # Frozen A transforms + training-only imputation/scaling.
    # ----------------------------------------------------------

    a_prep = NumericPreprocessor(
        a_features,
        {
            feature: A_TRANSFORM[feature]
            for feature in a_features
        },
    ).fit(train_q)

    A_train = a_prep.transform(
        train_q
    )
    A_target = a_prep.transform(
        target_q
    )

    a_support_cols = [
        A_SUPPORT[feature]
        for feature in a_features
    ]

    A_sup_train = get_binary_matrix(
        train_q,
        a_support_cols,
    )
    A_sup_target = get_binary_matrix(
        target_q,
        a_support_cols,
    )

    # ----------------------------------------------------------
    # Frozen Q transforms + training-only imputation/scaling.
    # QCHAN has already been rebuilt from training participants.
    # ----------------------------------------------------------

    q_prep = NumericPreprocessor(
        q_features,
        {
            feature: Q_TRANSFORM[feature]
            for feature in q_features
        },
    ).fit(train_q)

    Q_train = q_prep.transform(
        train_q
    )
    Q_target = q_prep.transform(
        target_q
    )

    q_support_cols = [
        column
        for column in Q_SUPPORT
        if column in train_q.columns
    ]

    Q_sup_train = get_binary_matrix(
        train_q,
        q_support_cols,
    )
    Q_sup_target = get_binary_matrix(
        target_q,
        q_support_cols,
    )

    # ----------------------------------------------------------
    # Age: complete case only, never imputed.
    # ----------------------------------------------------------

    age_train = pd.to_numeric(
        train_q[AGE],
        errors="coerce",
    ).to_numpy(float)[:, None]

    age_target = pd.to_numeric(
        target_q[AGE],
        errors="coerce",
    ).to_numpy(float)[:, None]

    if (
        not np.isfinite(age_train).all()
        or not np.isfinite(age_target).all()
    ):
        raise ValueError(
            "Primary Goal 2 received missing age."
        )

    # ----------------------------------------------------------
    # Training-only, support-aware linear Q->A residualization.
    # ----------------------------------------------------------

    residualizer = PerFeatureQResidualizer(
        alpha=RESIDUALIZER_ALPHA
    )

    residualizer.fit(
        Q_train,
        Q_sup_train,
        A_train,
        A_sup_train,
        train_q[
            "row_weight"
        ].to_numpy(float),
    )

    Ares_train = residualizer.transform(
        Q_train,
        Q_sup_train,
        A_train,
        A_sup_train,
    )

    Ares_target = residualizer.transform(
        Q_target,
        Q_sup_target,
        A_target,
        A_sup_target,
    )

    raw_designs_train = {
        "M_A": np.column_stack([
            age_train,
            A_train,
            A_sup_train,
        ]),
        "M_A+Q": np.column_stack([
            age_train,
            A_train,
            A_sup_train,
            Q_train,
            Q_sup_train,
        ]),
        "M_A-resQ": np.column_stack([
            age_train,
            Ares_train,
            A_sup_train,
        ]),
    }

    raw_designs_target = {
        "M_A": np.column_stack([
            age_target,
            A_target,
            A_sup_target,
        ]),
        "M_A+Q": np.column_stack([
            age_target,
            A_target,
            A_sup_target,
            Q_target,
            Q_sup_target,
        ]),
        "M_A-resQ": np.column_stack([
            age_target,
            Ares_target,
            A_sup_target,
        ]),
    }

    train_designs = {}
    target_designs = {}
    design_scalers = {}

    for model_name in [
        "M_A",
        "M_A+Q",
        "M_A-resQ",
    ]:
        scaler = FinalDesignScaler()

        train_designs[model_name] = (
            scaler.fit_transform(
                raw_designs_train[
                    model_name
                ]
            )
        )

        target_designs[model_name] = (
            scaler.transform(
                raw_designs_target[
                    model_name
                ]
            )
        )

        design_scalers[
            model_name
        ] = scaler

    metadata = {
        "a_preprocessor": a_prep,
        "q_preprocessor": q_prep,
        "residualizer": residualizer,
        "design_scalers": design_scalers,
        "train_with_qchan": train_q,
        "target_with_qchan": target_q,
        "q_support_cols": q_support_cols,
        "a_support_cols": a_support_cols,
    }

    return (
        train_designs,
        target_designs,
        metadata,
    )

print(
    "SUPPORT-AWARE PAIRED A / A+Q / A-resQ BUILDER: READY"
)


SUPPORT-AWARE PAIRED A / A+Q / A-resQ BUILDER: READY


## 9. Participant-grouped inner folds and paired ridge tuning

Diagnosis tuning minimizes participant-weighted log loss.

Severity tuning minimizes participant-weighted MAE.


In [10]:
def make_inner_participant_folds(
    train,
    task,
    seed,
):
    participant = (
        train[
            ["participant_id", "y"]
        ]
        .drop_duplicates(
            "participant_id"
        )
        .sort_values(
            "participant_id"
        )
        .reset_index(drop=True)
    )

    ids = participant[
        "participant_id"
    ].to_numpy()

    if task == "diagnosis":
        labels = participant[
            "y"
        ].to_numpy(int)

        splitter = StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=seed,
        )

        splits = splitter.split(
            ids,
            labels,
        )

    elif task == "severity":
        splitter = KFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=seed,
        )

        splits = splitter.split(ids)

    else:
        raise ValueError(task)

    output = []

    for train_idx, val_idx in splits:
        inner_train_ids = set(
            ids[train_idx].astype(str)
        )
        inner_val_ids = set(
            ids[val_idx].astype(str)
        )

        if inner_train_ids & inner_val_ids:
            raise RuntimeError(
                "Inner participant leakage."
            )

        output.append(
            (
                inner_train_ids,
                inner_val_ids,
            )
        )

    return output

def fit_outcome_model(
    X,
    y,
    weights,
    task,
    hyperparameter,
):
    if task == "diagnosis":
        model = LogisticRegression(
            C=float(hyperparameter),
            solver="lbfgs",
            max_iter=5000,
            class_weight=None,
        )

        model.fit(
            X,
            y.astype(int),
            sample_weight=weights,
        )

    else:
        model = Ridge(
            alpha=float(hyperparameter),
        )

        model.fit(
            X,
            y.astype(float),
            sample_weight=weights,
        )

    return model

def predict_outcome(
    model,
    X,
    task,
):
    if task == "diagnosis":
        return model.predict_proba(X)[:, 1]

    return model.predict(X)

def validation_loss(
    y,
    prediction,
    weights,
    task,
):
    if task == "diagnosis":
        return float(
            log_loss(
                y.astype(int),
                prediction,
                sample_weight=weights,
                labels=[0, 1],
            )
        )

    return float(
        np.average(
            np.abs(
                y.astype(float)
                - prediction
            ),
            weights=weights,
        )
    )

def tune_three_representations(
    train,
    task,
    seed,
    a_features=PRIMARY_A,
    q_features=None,
):
    if q_features is None:
        q_features = CORE_Q

    grid = (
        C_GRID
        if task == "diagnosis"
        else ALPHA_GRID
    )

    inner_folds = make_inner_participant_folds(
        train,
        task,
        seed,
    )

    fold_designs = []

    for fold_index, (
        inner_train_ids,
        inner_val_ids,
    ) in enumerate(
        inner_folds,
        start=1,
    ):
        inner_train = train.loc[
            train[
                "participant_id"
            ].isin(inner_train_ids)
        ].copy()

        inner_val = train.loc[
            train[
                "participant_id"
            ].isin(inner_val_ids)
        ].copy()

        # Recalculate participant-normalized weights inside this
        # exact inner fitting/validation population.
        inner_train["row_weight"] = participant_weights(
            inner_train
        )
        inner_val["row_weight"] = participant_weights(
            inner_val
        )

        train_designs, val_designs, _ = (
            fit_fold_representation(
                inner_train,
                inner_val,
                reference_ids=inner_train_ids,
                a_features=a_features,
                q_features=q_features,
            )
        )

        fold_designs.append({
            "train": inner_train,
            "val": inner_val,
            "X_train": train_designs,
            "X_val": val_designs,
        })

    selected = {}

    for representation in [
        "M_A",
        "M_A+Q",
        "M_A-resQ",
    ]:
        scores = []

        for hp in grid:
            fold_losses = []

            for item in fold_designs:
                y_train = item[
                    "train"
                ]["y"].to_numpy(float)

                w_train = item[
                    "train"
                ]["row_weight"].to_numpy(float)

                y_val = item[
                    "val"
                ]["y"].to_numpy(float)

                w_val = item[
                    "val"
                ]["row_weight"].to_numpy(float)

                model = fit_outcome_model(
                    item["X_train"][representation],
                    y_train,
                    w_train,
                    task,
                    hp,
                )

                pred = predict_outcome(
                    model,
                    item["X_val"][representation],
                    task,
                )

                fold_losses.append(
                    validation_loss(
                        y_val,
                        pred,
                        w_val,
                        task,
                    )
                )

            scores.append({
                "hyperparameter": float(hp),
                "mean_inner_loss": float(
                    np.mean(fold_losses)
                ),
            })

        score_table = pd.DataFrame(scores)

        best = score_table.sort_values(
            [
                "mean_inner_loss",
                "hyperparameter",
            ],
            ascending=[True, True],
        ).iloc[0]

        selected[representation] = float(
            best["hyperparameter"]
        )

    return selected

print("GROUPED INNER TUNING: READY")


GROUPED INNER TUNING: READY


## 10. Real-data one-fold preflight

This exercises the exact failure-prone path before the 50 outer folds begin:

- repeated diagnosis rows;
- repeated severity pairs;
- frozen A transforms/support;
- fold-safe QCHAN;
- training-only A/Q preprocessing;
- training-only Q→A residualization;
- grouped inner tuning;
- all three outcome models.


In [11]:
def preflight_branch(
    frame,
    task,
    label,
):
    manifest_slice = split_manifest.loc[
        split_manifest["repeat"].eq(1)
        & split_manifest["outer_fold"].eq(1)
    ]

    test_ids = set(
        manifest_slice[
            "participant_id"
        ].astype(str)
    )

    test_ids &= set(
        frame[
            "participant_id"
        ].astype(str)
    )

    train = frame.loc[
        ~frame[
            "participant_id"
        ].isin(test_ids)
    ].copy()

    test = frame.loc[
        frame[
            "participant_id"
        ].isin(test_ids)
    ].copy()

    if not len(train) or not len(test):
        raise RuntimeError(
            f"{label}: empty train/test preflight."
        )

    if set(
        train["participant_id"]
    ) & set(
        test["participant_id"]
    ):
        raise RuntimeError(
            f"{label}: participant leakage."
        )

    train["row_weight"] = participant_weights(
        train
    )
    test["row_weight"] = participant_weights(
        test
    )

    selected = tune_three_representations(
        train,
        task,
        BASE_SEED + 901,
    )

    train_designs, test_designs, meta = (
        fit_fold_representation(
            train,
            test,
            reference_ids=set(
                train["participant_id"]
            ),
            a_features=PRIMARY_A,
            q_features=CORE_Q,
        )
    )

    for representation in [
        "M_A",
        "M_A+Q",
        "M_A-resQ",
    ]:
        model = fit_outcome_model(
            train_designs[representation],
            train["y"].to_numpy(float),
            train["row_weight"].to_numpy(float),
            task,
            selected[representation],
        )

        pred = predict_outcome(
            model,
            test_designs[representation],
            task,
        )

        if (
            len(pred) != len(test)
            or not np.isfinite(pred).all()
        ):
            raise RuntimeError(
                f"{label}: invalid predictions for "
                f"{representation}."
            )

    print(
        f"{label}: PASS | "
        f"train participants={train['participant_id'].nunique()} | "
        f"test participants={test['participant_id'].nunique()}"
    )

preflight_branch(
    dx,
    "diagnosis",
    "Diagnosis Goal 2 preflight",
)

preflight_branch(
    sev,
    "severity",
    "Severity Goal 2 preflight",
)

print("REAL-DATA GOAL 2 PREFLIGHT: PASS")


Diagnosis Goal 2 preflight: PASS | train participants=161 | test participants=38
Severity Goal 2 preflight: PASS | train participants=116 | test participants=29
REAL-DATA GOAL 2 PREFLIGHT: PASS


## 11. Goal 2 completion output space and FINAL primary-run gate

Notebook 20 is treated as an immutable input. All corrected/robustness outputs below are written to a new directory. The original FINAL files are never overwritten.


In [12]:

PRIMARY_OUT = OUT
PRIMARY_TABLES = TABLES
PRIMARY_OOF_DIR = OOF_DIR

primary_success_path = PRIMARY_OUT / "SUCCESS.json"
if not primary_success_path.exists():
    raise FileNotFoundError(
        "Notebook 20 FINAL SUCCESS.json is missing. Run Notebook 20 in FINAL mode first."
    )
primary_success = json.loads(primary_success_path.read_text(encoding="utf-8"))
if primary_success.get("status") != "PASS" or primary_success.get("run_mode") != "FINAL":
    raise RuntimeError("Notebook 20 primary run is not a sealed FINAL PASS.")
if int(primary_success.get("bootstrap_replicates", -1)) != 2000:
    raise RuntimeError("Notebook 20 FINAL did not use 2,000 participant bootstraps.")

primary_dx_path = PRIMARY_OOF_DIR / "goal2_diagnosis_oof.csv"
primary_sev_path = PRIMARY_OOF_DIR / "goal2_severity_oof.csv"
primary_manifest_path = PRIMARY_TABLES / "goal2_fold_model_manifest.csv"
for p in [primary_dx_path, primary_sev_path, primary_manifest_path]:
    if not p.exists() or p.stat().st_size == 0:
        raise FileNotFoundError(f"Required Notebook 20 FINAL artifact missing/empty: {p}")

primary_dx_oof = safe_read_csv(
    primary_dx_path,
    required_columns=["participant_id","logical_recording_id","repeat","outer_fold","model","y","prediction","row_weight"],
)
primary_sev_oof = safe_read_csv(
    primary_sev_path,
    required_columns=["participant_id","logical_recording_id","repeat","outer_fold","model","y","prediction","row_weight"],
)
primary_fold_manifest = safe_read_csv(
    primary_manifest_path,
    required_columns=["task","repeat","outer_fold","model","selected_hyperparameter"],
)

expected_models = {"M_A", "M_A+Q", "M_A-resQ"}
for label, frame in [("diagnosis", primary_dx_oof), ("severity", primary_sev_oof)]:
    if set(frame["model"].astype(str)) != expected_models:
        raise RuntimeError(f"{label}: primary OOF model set changed.")
    if frame["repeat"].nunique() != 10:
        raise RuntimeError(f"{label}: primary OOF is not complete across 10 repeats.")

# New, non-destructive completion output tree.
OUT = ROOT / "outputs" / "goal2" / "goal2_completion_v1_0" / "final"
TABLES = OUT / "tables"
FIGURES = OUT / "figures"
OOF_DIR = OUT / "oof"
CHECKPOINTS = OUT / "checkpoints"
LOGS = OUT / "logs"
for directory in [OUT, TABLES, FIGURES, OOF_DIR, CHECKPOINTS, LOGS]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_HGB = True
N_COMPLETION_BOOTSTRAPS = 2000
HGB_MAX_ITER = 200
HGB_GRID = [
    {
        "max_depth": depth,
        "learning_rate": lr,
        "min_samples_leaf": leaf,
        "l2_regularization": l2,
    }
    for depth, lr, leaf, l2 in itertools.product(
        [2, 3], [0.03, 0.10], [15, 30], [0.0, 1.0]
    )
]
RESIDUALIZER_ALPHA_GRID = np.logspace(-4, 4, 9)

print("NOTEBOOK 20 FINAL INPUT: VERIFIED")
print("Primary output:", PRIMARY_OUT)
print("Completion output:", OUT)
print("HGB sensitivity:", RUN_HGB)


NOTEBOOK 20 FINAL INPUT: VERIFIED
Primary output: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal2\goal2_primary_v1_1\final
Completion output: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal2\goal2_completion_v1_0\final
HGB sensitivity: True


## 12. Protocol-concordance audit

The frozen Methods state that the linear Q→A residualizer penalty is selected from `10^-4 ... 10^4` by participant-grouped inner CV minimizing standardized acoustic reconstruction error. Notebook 20 instead fixed this penalty at 1.0. Because this affects only `M_A-resQ`, this notebook corrects that representation without touching the already-valid `M_A` or `M_A+Q` OOF predictions.

The correction is mechanical and protocol-driven, not selected according to the observed clinical direction.


In [13]:

protocol_audit = pd.DataFrame([
    {
        "item": "M_A and M_A+Q fold-safe preprocessing/tuning",
        "notebook20": "implemented",
        "frozen_methods": "implemented",
        "action": "retain Notebook 20 FINAL OOF",
        "status": "concordant",
    },
    {
        "item": "Q->A residualizer penalty",
        "notebook20": f"fixed alpha={RESIDUALIZER_ALPHA:g}",
        "frozen_methods": "grouped inner-CV alpha in 1e-4...1e4 using reconstruction error",
        "action": "rerun only M_A-resQ with protocol-concordant alpha selection",
        "status": "correction_required",
    },
])
atomic_write_csv(protocol_audit, TABLES / "goal2_protocol_concordance_audit.csv")
display(protocol_audit)
print("PROTOCOL-CONCORDANCE AUDIT: COMPLETE")


,item,notebook20,frozen_methods,action,status
0,M_A and M_A+Q fold-safe preprocessing/tuning,implemented,implemented,retain Notebook 20 FINAL OOF,concordant
1,Q->A residualizer penalty,fixed alpha=1,grouped inner-CV alpha in 1e-4...1e4 using rec...,rerun only M_A-resQ with protocol-concordant a...,correction_required


PROTOCOL-CONCORDANCE AUDIT: COMPLETE


## 13. Generalized fold-safe design builder

This builder reuses the validated Notebook 20 QCHAN reconstruction and preprocessing machinery, but makes Q support handling explicit. It is used for the corrected residualized model and all sensitivity analyses.


In [14]:

EXTENDED_Q_EXTRA = [
    "qadd_pause_spectral_flatness",
    "qadd_mains_hum_comb_score_db",
    "qrev_tail_excess_100ms_db",
    "qrev_tail_persistence_median_sec",
    "qrev_downward_decay_rate_db_per_sec",
]
EXTENDED_Q = CORE_Q + EXTENDED_Q_EXTRA
for feature in EXTENDED_Q:
    if feature not in recording_table.columns and feature not in QCHAN:
        raise KeyError(f"Extended-Q feature missing from canonical recording table: {feature}")
    if feature not in Q_TRANSFORM:
        raise KeyError(f"Extended-Q transform missing from q_registry: {feature}")

PERSISTENCE_CENSOR = "qrev_persistence_recording_median_censored"
if PERSISTENCE_CENSOR not in recording_table.columns:
    status_col = "qrev_tail_persistence_median_sec_status"
    if status_col not in recording_table.columns:
        raise KeyError(
            "Extended-Q requires a QREV persistence censor indicator or its registered status field."
        )
    recording_table[PERSISTENCE_CENSOR] = (
        recording_table[status_col].astype(str)
        .str.contains("censor", case=False, na=False)
        .astype(float)
    )

# Propagate the censor variable into model rows by recording identity.
censor_lookup = recording_table[["logical_recording_id", PERSISTENCE_CENSOR]].drop_duplicates()
if censor_lookup["logical_recording_id"].duplicated().any():
    raise RuntimeError("Persistence censor lookup is not one row per recording.")

def attach_persistence_censor(frame):
    return frame.drop(columns=[PERSISTENCE_CENSOR], errors="ignore").merge(
        censor_lookup,
        on="logical_recording_id",
        how="left",
        validate="many_to_one",
    )

dx = attach_persistence_censor(dx)
sev = attach_persistence_censor(sev)

Q_FAMILIES = OrderedDict([
    ("QADD", QADD),
    ("QGAIN", QGAIN),
    ("QREV", QREV),
    ("QCHAN", QCHAN),
])

CORE_SUPPORT_SOURCES = list(QADD)
EXTENDED_SUPPORT_SOURCES = list(QADD) + list(EXTENDED_Q_EXTRA)


def _with_support_columns(frame, support_sources):
    out = frame.copy()
    cols = []
    for feature in support_sources:
        if feature not in out.columns:
            raise KeyError(f"Cannot create support indicator; feature missing: {feature}")
        col = f"__q_support__{feature}"
        out[col] = out[feature].notna().astype(float)
        cols.append(col)
        legacy = f"{feature}_supported"
        if legacy in out.columns:
            legacy_values = pd.to_numeric(out[legacy], errors="coerce").to_numpy(float)
            if not np.array_equal(legacy_values, out[col].to_numpy(float)):
                raise RuntimeError(f"Frozen support semantics changed for {feature}.")
    return out, cols


def make_spec(
    name,
    *,
    q_features=(),
    q_support_sources=(),
    q_binary_cols=(),
    include_age=True,
    include_sex=False,
    residualize=False,
):
    return {
        "name": str(name),
        "q_features": list(q_features),
        "q_support_sources": list(q_support_sources),
        "q_binary_cols": list(q_binary_cols),
        "include_age": bool(include_age),
        "include_sex": bool(include_sex),
        "residualize": bool(residualize),
    }

PRIMARY_SPECS = {
    "M_A": make_spec("M_A"),
    "M_A+Q": make_spec(
        "M_A+Q", q_features=CORE_Q, q_support_sources=CORE_SUPPORT_SOURCES
    ),
    "M_A-resQ": make_spec(
        "M_A-resQ", q_features=CORE_Q, q_support_sources=CORE_SUPPORT_SOURCES,
        residualize=True,
    ),
}


def fit_design_spec(train, target, spec, *, reference_ids, residualizer_alpha=None):
    train_local = train.copy()
    target_local = target.copy()

    q_features = list(spec["q_features"])
    needs_qchan = any(f in QCHAN for f in q_features)
    if needs_qchan:
        train_local = attach_fold_safe_qchan(train_local, reference_ids)
        target_local = attach_fold_safe_qchan(target_local, reference_ids)

    # A: same frozen Primary-A in every Goal 2 representation.
    a_prep = NumericPreprocessor(
        PRIMARY_A,
        {feature: A_TRANSFORM[feature] for feature in PRIMARY_A},
    ).fit(train_local)
    A_train = a_prep.transform(train_local)
    A_target = a_prep.transform(target_local)
    a_support_cols = [A_SUPPORT[f] for f in PRIMARY_A]
    A_sup_train = get_binary_matrix(train_local, a_support_cols)
    A_sup_target = get_binary_matrix(target_local, a_support_cols)

    # Q, with explicit support/censor indicators only when part of the spec.
    if q_features:
        train_local, q_support_cols = _with_support_columns(
            train_local, spec["q_support_sources"]
        )
        target_local, q_support_cols_target = _with_support_columns(
            target_local, spec["q_support_sources"]
        )
        if q_support_cols != q_support_cols_target:
            raise RuntimeError("Q support column mismatch train/target.")

        q_prep = NumericPreprocessor(
            q_features,
            {feature: Q_TRANSFORM[feature] for feature in q_features},
        ).fit(train_local)
        Q_train = q_prep.transform(train_local)
        Q_target = q_prep.transform(target_local)
        Q_sup_train = get_binary_matrix(train_local, q_support_cols)
        Q_sup_target = get_binary_matrix(target_local, q_support_cols)

        binary_cols = list(spec["q_binary_cols"])
        Q_bin_train = get_binary_matrix(train_local, binary_cols)
        Q_bin_target = get_binary_matrix(target_local, binary_cols)
        Q_aux_train = np.column_stack([Q_sup_train, Q_bin_train])
        Q_aux_target = np.column_stack([Q_sup_target, Q_bin_target])
    else:
        q_prep = None
        Q_train = np.empty((len(train_local), 0), float)
        Q_target = np.empty((len(target_local), 0), float)
        Q_aux_train = np.empty((len(train_local), 0), float)
        Q_aux_target = np.empty((len(target_local), 0), float)
        q_support_cols = []
        binary_cols = []

    cov_train = []
    cov_target = []
    cov_names = []
    if spec["include_age"]:
        age_train = pd.to_numeric(train_local[AGE], errors="coerce").to_numpy(float)[:, None]
        age_target = pd.to_numeric(target_local[AGE], errors="coerce").to_numpy(float)[:, None]
        if not np.isfinite(age_train).all() or not np.isfinite(age_target).all():
            raise ValueError(f"{spec['name']}: age is missing in a complete-case analysis.")
        cov_train.append(age_train); cov_target.append(age_target); cov_names.append("Age")
    if spec["include_sex"]:
        sex_train = pd.to_numeric(train_local["sex_binary"], errors="coerce").to_numpy(float)[:, None]
        sex_target = pd.to_numeric(target_local["sex_binary"], errors="coerce").to_numpy(float)[:, None]
        if not np.isfinite(sex_train).all() or not np.isfinite(sex_target).all():
            raise ValueError(f"{spec['name']}: sex is missing in complete-case sensitivity.")
        cov_train.append(sex_train); cov_target.append(sex_target); cov_names.append("Sex")

    C_train = np.column_stack(cov_train) if cov_train else np.empty((len(train_local), 0), float)
    C_target = np.column_stack(cov_target) if cov_target else np.empty((len(target_local), 0), float)

    residualizer = None
    if spec["residualize"]:
        if not q_features:
            raise ValueError("Residualized representation requires Q features.")
        if residualizer_alpha is None:
            raise ValueError("Residualized representation requires a selected residualizer alpha.")
        residualizer = PerFeatureQResidualizer(alpha=float(residualizer_alpha))
        residualizer.fit(
            Q_train, Q_aux_train, A_train, A_sup_train,
            train_local["row_weight"].to_numpy(float),
        )
        A_used_train = residualizer.transform(
            Q_train, Q_aux_train, A_train, A_sup_train
        )
        A_used_target = residualizer.transform(
            Q_target, Q_aux_target, A_target, A_sup_target
        )
        a_names = [f"{f}_resQ" for f in PRIMARY_A]
    else:
        A_used_train = A_train
        A_used_target = A_target
        a_names = list(PRIMARY_A)

    raw_train_parts = [C_train, A_used_train, A_sup_train]
    raw_target_parts = [C_target, A_used_target, A_sup_target]
    names = cov_names + a_names + [f"support:{c}" for c in a_support_cols]

    # Q enters the clinical design only when explicit, not when used solely for residualization.
    if q_features and not spec["residualize"]:
        raw_train_parts.extend([Q_train, Q_aux_train])
        raw_target_parts.extend([Q_target, Q_aux_target])
        names.extend(q_features)
        names.extend([f"support:{c}" for c in q_support_cols])
        names.extend([f"binary:{c}" for c in binary_cols])

    raw_train = np.column_stack(raw_train_parts)
    raw_target = np.column_stack(raw_target_parts)
    scaler = FinalDesignScaler()
    X_train = scaler.fit_transform(raw_train)
    X_target = scaler.transform(raw_target)

    if not np.isfinite(X_train).all() or not np.isfinite(X_target).all():
        raise RuntimeError(f"{spec['name']}: nonfinite design after fold-safe preprocessing.")

    return X_train, X_target, {
        "feature_names": names,
        "train_with_qchan": train_local,
        "target_with_qchan": target_local,
        "A_target": A_target,
        "A_support_target": A_sup_target,
        "A_used_target": A_used_target,
        "residualizer": residualizer,
    }

print("GENERALIZED FOLD-SAFE DESIGN BUILDER: READY")


GENERALIZED FOLD-SAFE DESIGN BUILDER: READY


## 14. Protocol-concordant Q→A residualizer tuning

The residualizer alpha is selected using **only outer-training participants**, by grouped inner CV and weighted standardized reconstruction MSE across the six Primary-A features. No diagnosis/bulbar prediction performance is used to select this alpha.


In [15]:

def reconstruction_loss_for_fold(inner_train, inner_val, spec, alpha):
    inner_train = inner_train.copy(); inner_val = inner_val.copy()
    inner_train["row_weight"] = participant_weights(inner_train)
    inner_val["row_weight"] = participant_weights(inner_val)
    Xtr, Xv, meta = fit_design_spec(
        inner_train, inner_val, spec,
        reference_ids=set(inner_train["participant_id"].astype(str)),
        residualizer_alpha=float(alpha),
    )
    Ares = meta["A_used_target"]
    Asup = meta["A_support_target"]
    w = inner_val["row_weight"].to_numpy(float)
    per_feature = []
    for j in range(Ares.shape[1]):
        mask = Asup[:, j] == 1
        if mask.sum() < 5:
            continue
        per_feature.append(float(np.average(Ares[mask, j] ** 2, weights=w[mask])))
    if not per_feature:
        raise RuntimeError("No supported A feature available for residualizer reconstruction scoring.")
    return float(np.mean(per_feature))


def select_residualizer_alpha(train, task, seed, spec):
    if not spec["residualize"]:
        return None, pd.DataFrame()
    folds = make_inner_participant_folds(train, task, seed)
    rows = []
    for alpha in RESIDUALIZER_ALPHA_GRID:
        losses = []
        for fold_idx, (tr_ids, va_ids) in enumerate(folds, start=1):
            tr = train.loc[train["participant_id"].isin(tr_ids)].copy()
            va = train.loc[train["participant_id"].isin(va_ids)].copy()
            loss = reconstruction_loss_for_fold(tr, va, spec, alpha)
            losses.append(loss)
            rows.append({
                "alpha": float(alpha), "inner_fold": fold_idx,
                "reconstruction_mse": float(loss),
            })
    summary = (
        pd.DataFrame(rows).groupby("alpha", as_index=False)["reconstruction_mse"]
        .mean().sort_values(["reconstruction_mse", "alpha"], kind="stable")
    )
    selected = float(summary.iloc[0]["alpha"])
    return selected, pd.DataFrame(rows)


def tune_ridge_spec(train, task, seed, spec, residualizer_alpha=None):
    grid = C_GRID if task == "diagnosis" else ALPHA_GRID
    folds = make_inner_participant_folds(train, task, seed)
    bundles = []
    for fold_idx, (tr_ids, va_ids) in enumerate(folds, start=1):
        tr = train.loc[train["participant_id"].isin(tr_ids)].copy()
        va = train.loc[train["participant_id"].isin(va_ids)].copy()
        tr["row_weight"] = participant_weights(tr)
        va["row_weight"] = participant_weights(va)
        Xtr, Xv, _ = fit_design_spec(
            tr, va, spec,
            reference_ids=set(tr["participant_id"].astype(str)),
            residualizer_alpha=residualizer_alpha,
        )
        bundles.append((tr, va, Xtr, Xv))

    score_rows = []
    for param in grid:
        losses = []
        for fold_idx, (tr, va, Xtr, Xv) in enumerate(bundles, start=1):
            model = fit_outcome_model(
                Xtr, tr["y"].to_numpy(float), tr["row_weight"].to_numpy(float),
                task, float(param),
            )
            pred = predict_outcome(model, Xv, task)
            losses.append(validation_loss(
                va["y"].to_numpy(float), pred,
                va["row_weight"].to_numpy(float), task,
            ))
        score_rows.append({"parameter": float(param), "mean_inner_loss": float(np.mean(losses))})
    scores = pd.DataFrame(score_rows).sort_values(["mean_inner_loss", "parameter"], kind="stable")
    return float(scores.iloc[0]["parameter"]), scores

print("RESIDUALIZER + OUTCOME TUNING: READY")


RESIDUALIZER + OUTCOME TUNING: READY


## 15. Generic outer-CV runner with per-fold checkpoints

This runner is used for the corrected residualized model and ridge sensitivities. Every checkpoint is one outer fold, so interruptions do not destroy completed work.


In [16]:

def _slug(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_")


def run_ridge_spec_cv(frame, task, spec, *, label):
    frame = frame.copy()
    out_dir = CHECKPOINTS / _slug(label) / task
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_rows = []
    fold_frames = []

    for repeat in range(1, OUTER_REPEATS + 1):
        print(f"{label} | {task} | repeat {repeat}/{OUTER_REPEATS}")
        for outer_fold in range(1, OUTER_FOLDS + 1):
            fold_path = out_dir / f"r{repeat:02d}_f{outer_fold:02d}.csv"
            meta_path = out_dir / f"r{repeat:02d}_f{outer_fold:02d}.json"
            if RESUME_IF_VALID and fold_path.exists() and meta_path.exists() and fold_path.stat().st_size > 0:
                f = safe_read_csv(fold_path, required_columns=[
                    "participant_id","logical_recording_id","repeat","outer_fold","model","y","prediction","row_weight"
                ])
                m = json.loads(meta_path.read_text(encoding="utf-8"))
                fold_frames.append(f); manifest_rows.append(m)
                continue

            test_ids = set(split_manifest.loc[
                split_manifest["repeat"].eq(repeat) & split_manifest["outer_fold"].eq(outer_fold),
                "participant_id",
            ].astype(str)) & set(frame["participant_id"].astype(str))
            train = frame.loc[~frame["participant_id"].isin(test_ids)].copy()
            test = frame.loc[frame["participant_id"].isin(test_ids)].copy()
            if not len(train) or not len(test):
                raise RuntimeError(f"{label}/{task}/r{repeat}/f{outer_fold}: empty split.")
            if set(train["participant_id"]) & set(test["participant_id"]):
                raise RuntimeError("Outer participant leakage detected.")
            train["row_weight"] = participant_weights(train)
            test["row_weight"] = participant_weights(test)
            seed = deterministic_seed("goal2_completion", label, task, repeat, outer_fold)

            selected_resid = None
            resid_scores = pd.DataFrame()
            if spec["residualize"]:
                selected_resid, resid_scores = select_residualizer_alpha(train, task, seed + 17, spec)
            selected_outcome, outcome_scores = tune_ridge_spec(
                train, task, seed + 31, spec, residualizer_alpha=selected_resid
            )
            Xtr, Xte, meta = fit_design_spec(
                train, test, spec,
                reference_ids=set(train["participant_id"].astype(str)),
                residualizer_alpha=selected_resid,
            )
            model = fit_outcome_model(
                Xtr, train["y"].to_numpy(float), train["row_weight"].to_numpy(float),
                task, selected_outcome,
            )
            pred = predict_outcome(model, Xte, task)
            result_cols = ["participant_id","logical_recording_id","y","row_weight"]
            if task == "severity":
                for c in ["assessment_date","abs_delta_days"]:
                    if c in test.columns:
                        result_cols.append(c)
            result = test[result_cols].copy()
            result["repeat"] = repeat
            result["outer_fold"] = outer_fold
            result["model"] = spec["name"]
            result["prediction"] = pred
            result["selected_hyperparameter"] = selected_outcome
            result["selected_residualizer_alpha"] = np.nan if selected_resid is None else selected_resid
            result["loss"] = (
                (result["y"] - result["prediction"]) ** 2
                if task == "diagnosis"
                else np.abs(result["y"] - result["prediction"])
            )
            atomic_write_csv(result, fold_path)
            fold_meta = {
                "label": label, "task": task, "repeat": repeat, "outer_fold": outer_fold,
                "model": spec["name"], "selected_hyperparameter": float(selected_outcome),
                "selected_residualizer_alpha": None if selected_resid is None else float(selected_resid),
                "n_train_participants": int(train["participant_id"].nunique()),
                "n_test_participants": int(test["participant_id"].nunique()),
            }
            atomic_write_json(fold_meta, meta_path)
            fold_frames.append(result); manifest_rows.append(fold_meta)

    oof = pd.concat(fold_frames, ignore_index=True)
    if oof["repeat"].nunique() != OUTER_REPEATS:
        raise RuntimeError(f"{label}/{task}: incomplete repeat coverage.")
    key = ["participant_id","logical_recording_id","repeat","model"]
    if task == "severity" and "assessment_date" in oof.columns:
        key.append("assessment_date")
    if oof.groupby(key).size().ne(1).any():
        raise RuntimeError(f"{label}/{task}: duplicate or missing OOF identity.")
    return oof, pd.DataFrame(manifest_rows)

print("CHECKPOINTED RIDGE OUTER-CV RUNNER: READY")


CHECKPOINTED RIDGE OUTER-CV RUNNER: READY


## 16. Preflight the corrected residualized pathway on one outer fold

No full correction run begins until a real-data fold completes all nested residualizer and outcome tuning without leakage or nonfinite predictions.


In [17]:

def preflight_corrected_resq(frame, task):
    repeat, outer_fold = 1, 1
    test_ids = set(split_manifest.loc[
        split_manifest["repeat"].eq(repeat) & split_manifest["outer_fold"].eq(outer_fold),
        "participant_id",
    ].astype(str)) & set(frame["participant_id"].astype(str))
    train = frame.loc[~frame["participant_id"].isin(test_ids)].copy()
    test = frame.loc[frame["participant_id"].isin(test_ids)].copy()
    train["row_weight"] = participant_weights(train); test["row_weight"] = participant_weights(test)
    seed = deterministic_seed("preflight_corrected_resq", task)
    alpha, alpha_scores = select_residualizer_alpha(train, task, seed, PRIMARY_SPECS["M_A-resQ"])
    param, param_scores = tune_ridge_spec(train, task, seed+1, PRIMARY_SPECS["M_A-resQ"], residualizer_alpha=alpha)
    Xtr, Xte, _ = fit_design_spec(
        train, test, PRIMARY_SPECS["M_A-resQ"],
        reference_ids=set(train["participant_id"].astype(str)), residualizer_alpha=alpha,
    )
    model = fit_outcome_model(Xtr, train["y"].to_numpy(float), train["row_weight"].to_numpy(float), task, param)
    pred = predict_outcome(model, Xte, task)
    if len(pred) != len(test) or not np.isfinite(pred).all():
        raise RuntimeError(f"Corrected {task} preflight produced invalid predictions.")
    return {"task":task,"selected_residualizer_alpha":alpha,"selected_outcome_parameter":param,
            "n_train":train["participant_id"].nunique(),"n_test":test["participant_id"].nunique()}

preflight_table = pd.DataFrame([
    preflight_corrected_resq(dx, "diagnosis"),
    preflight_corrected_resq(sev, "severity"),
])
atomic_write_csv(preflight_table, TABLES / "goal2_completion_preflight.csv")
display(preflight_table)
print("CORRECTED M_A-resQ PREFLIGHT: PASS")


,task,selected_residualizer_alpha,selected_outcome_parameter,n_train,n_test
0,diagnosis,10.0,0.1000,161,38
1,severity,10.0,0.0001,116,29


CORRECTED M_A-resQ PREFLIGHT: PASS


## 17. Run the protocol-concordant `M_A-resQ` and build the authoritative Goal 2 OOF objects

`M_A` and `M_A+Q` are taken directly from the sealed Notebook 20 FINAL run. Only `M_A-resQ` is regenerated.


In [18]:

corrected_dx_resq, corrected_dx_manifest = run_ridge_spec_cv(
    dx, "diagnosis", PRIMARY_SPECS["M_A-resQ"], label="corrected_primary_M_A-resQ"
)
corrected_sev_resq, corrected_sev_manifest = run_ridge_spec_cv(
    sev, "severity", PRIMARY_SPECS["M_A-resQ"], label="corrected_primary_M_A-resQ"
)

# Preserve exact Q columns from Notebook 20 for downstream natural-Q analyses.
def attach_primary_q_columns(corrected, primary, task):
    qcols = [q for q in CORE_Q + Q_SUPPORT if q in primary.columns]
    keys = ["participant_id","logical_recording_id","repeat","outer_fold"]
    if task == "severity" and "assessment_date" in corrected.columns and "assessment_date" in primary.columns:
        keys.append("assessment_date")
    lookup = (
        primary.loc[primary["model"].eq("M_A"), keys + qcols]
        .drop_duplicates(keys)
    )
    out = corrected.merge(lookup, on=keys, how="left", validate="one_to_one")
    if qcols and out[qcols].isna().all(axis=None):
        raise RuntimeError("Corrected OOF failed to inherit fold-safe Core-Q values.")
    return out

corrected_dx_resq = attach_primary_q_columns(corrected_dx_resq, primary_dx_oof, "diagnosis")
corrected_sev_resq = attach_primary_q_columns(corrected_sev_resq, primary_sev_oof, "severity")

dx_oof = pd.concat([
    primary_dx_oof.loc[primary_dx_oof["model"].isin(["M_A","M_A+Q"])].copy(),
    corrected_dx_resq.copy(),
], ignore_index=True)
sev_oof = pd.concat([
    primary_sev_oof.loc[primary_sev_oof["model"].isin(["M_A","M_A+Q"])].copy(),
    corrected_sev_resq.copy(),
], ignore_index=True)

for label, frame in [("diagnosis", dx_oof), ("severity", sev_oof)]:
    if set(frame["model"].astype(str)) != expected_models:
        raise RuntimeError(f"{label}: authoritative model set incomplete.")
    if frame["repeat"].nunique() != 10:
        raise RuntimeError(f"{label}: authoritative OOF repeat coverage incomplete.")

atomic_write_csv(dx_oof, OOF_DIR / "goal2_diagnosis_oof_authoritative.csv")
atomic_write_csv(sev_oof, OOF_DIR / "goal2_severity_oof_authoritative.csv")
corrected_manifest = pd.concat([corrected_dx_manifest, corrected_sev_manifest], ignore_index=True)
atomic_write_csv(corrected_manifest, TABLES / "goal2_corrected_residualizer_fold_manifest.csv")

print("AUTHORITATIVE GOAL 2 OOF: READY")
print("Diagnosis rows:", len(dx_oof), "| Severity rows:", len(sev_oof))


NameError: name 're' is not defined

## 18. Recompute authoritative metrics, calibration, 2,000 paired bootstrap CIs, and individual shifts

In [ ]:

MODELS = ["M_A","M_A+Q","M_A-resQ"]

def participant_dx_by_repeat_local(oof, agg="mean"):
    rows=[]
    for (model,repeat,pid),g in oof.groupby(["model","repeat","participant_id"]):
        yv=g["y"].unique()
        if len(yv)!=1: raise RuntimeError("Diagnosis varies within participant.")
        pred = float(g["prediction"].mean() if agg=="mean" else g["prediction"].median())
        rows.append({"model":model,"repeat":int(repeat),"participant_id":pid,"y":float(yv[0]),"prediction":pred})
    return pd.DataFrame(rows)

def diagnosis_repeat_metrics_local(pr):
    rows=[]
    for (model,repeat),g in pr.groupby(["model","repeat"]):
        y=g["y"].to_numpy(int); p=g["prediction"].to_numpy(float)
        intercept,slope=safe_calibration_stats(y,p)
        rows.append({"task":"diagnosis","model":model,"repeat":int(repeat),
                     "AUROC":roc_auc_score(y,p),"Brier":brier_score_loss(y,p),
                     "AUPRC":average_precision_score(y,p),"CalibrationIntercept":intercept,
                     "CalibrationSlope":slope})
    return pd.DataFrame(rows)

def severity_repeat_metrics_local(oof):
    rows=[]
    for (model,repeat),g in oof.groupby(["model","repeat"]):
        y=g["y"].to_numpy(float); p=g["prediction"].to_numpy(float); w=g["row_weight"].to_numpy(float)
        rows.append({"task":"severity","model":model,"repeat":int(repeat),
                     "MAE":float(np.average(np.abs(y-p),weights=w)),
                     "RMSE":float(np.sqrt(np.average((y-p)**2,weights=w))),
                     "Spearman_rho":float(stats.spearmanr(y,p,nan_policy="omit").statistic),
                     "R2":float(r2_score(y,p,sample_weight=w))})
    return pd.DataFrame(rows)

def safe_calibration_stats(y,p):
    y=np.asarray(y,float); p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    if len(np.unique(y))<2: return np.nan,np.nan
    lp=np.log(p/(1-p))
    try:
        intercept=float(sm.GLM(y,np.ones((len(y),1)),family=sm.families.Binomial(),offset=lp).fit().params[0])
    except Exception: intercept=np.nan
    try:
        slope=float(sm.GLM(y,sm.add_constant(lp),family=sm.families.Binomial()).fit().params[1])
    except Exception: slope=np.nan
    return intercept,slope

dx_participant_repeat = participant_dx_by_repeat_local(dx_oof,"mean")
dx_metrics_repeat = diagnosis_repeat_metrics_local(dx_participant_repeat)
sev_metrics_repeat = severity_repeat_metrics_local(sev_oof)
atomic_write_csv(dx_participant_repeat,TABLES/"goal2_dx_participant_oof_by_repeat_authoritative.csv")
atomic_write_csv(dx_metrics_repeat,TABLES/"goal2_dx_metrics_by_repeat_authoritative.csv")
atomic_write_csv(sev_metrics_repeat,TABLES/"goal2_severity_metrics_by_repeat_authoritative.csv")

summary=[]
for model in MODELS:
    d=dx_metrics_repeat.loc[dx_metrics_repeat["model"].eq(model)]
    s=sev_metrics_repeat.loc[sev_metrics_repeat["model"].eq(model)]
    for metric in ["AUROC","Brier","AUPRC","CalibrationIntercept","CalibrationSlope"]:
        summary.append({"task":"diagnosis","model":model,"metric":metric,"estimate":float(d[metric].mean())})
    for metric in ["MAE","RMSE","Spearman_rho","R2"]:
        summary.append({"task":"severity","model":model,"metric":metric,"estimate":float(s[metric].mean())})
authoritative_metrics=pd.DataFrame(summary)
atomic_write_csv(authoritative_metrics,TABLES/"goal2_authoritative_observed_metrics.csv")

# Vectorized participant-cluster paired bootstrap. This avoids pandas duplicate-cluster internals.
def bootstrap_contrast_diagnosis(pr, model_b, model_a, metric, B, seed):
    participants=np.array(sorted(pr["participant_id"].astype(str).unique()))
    repeats=np.array(sorted(pr["repeat"].unique()))
    lookup={}
    for model in [model_a,model_b]:
        m=pr.loc[pr["model"].eq(model)].copy()
        for r in repeats:
            g=m.loc[m["repeat"].eq(r)].set_index("participant_id").reindex(participants)
            lookup[(model,r)]=(g["y"].to_numpy(int),g["prediction"].to_numpy(float))
    rng=np.random.default_rng(seed); draws=np.empty(B,float)
    def metric_fn(y,p):
        if metric=="AUROC": return roc_auc_score(y,p)
        if metric=="Brier": return brier_score_loss(y,p)
        if metric in {"CalibrationIntercept","CalibrationSlope"}:
            i,s=safe_calibration_stats(y,p); return i if metric=="CalibrationIntercept" else s
        raise ValueError(metric)
    for b in range(B):
        idx=rng.integers(0,len(participants),size=len(participants))
        vals=[]
        for r in repeats:
            ya,pa=lookup[(model_a,r)]; yb,pb=lookup[(model_b,r)]
            vals.append(metric_fn(yb[idx],pb[idx])-metric_fn(ya[idx],pa[idx]))
        draws[b]=np.mean(vals)
    obs=[]
    for r in repeats:
        ya,pa=lookup[(model_a,r)]; yb,pb=lookup[(model_b,r)]
        obs.append(metric_fn(yb,pb)-metric_fn(ya,pa))
    return float(np.mean(obs)),draws

def _severity_participant_contrib(oof, model, metric):
    rows=[]
    for (repeat,pid),g in oof.loc[oof["model"].eq(model)].groupby(["repeat","participant_id"]):
        y=g["y"].to_numpy(float); p=g["prediction"].to_numpy(float)
        val=float(np.mean(np.abs(y-p))) if metric=="MAE" else float(np.sqrt(np.mean((y-p)**2)))
        # For RMSE we cannot average participant RMSE to equal pooled participant-weighted RMSE.
        # Store MSE contribution and transform after averaging participants.
        if metric=="RMSE": val=float(np.mean((y-p)**2))
        rows.append({"repeat":repeat,"participant_id":str(pid),"value":val})
    return pd.DataFrame(rows)

def bootstrap_contrast_severity(oof, model_b, model_a, metric, B, seed):
    participants=np.array(sorted(oof["participant_id"].astype(str).unique()))
    repeats=np.array(sorted(oof["repeat"].unique()))
    lookup={}
    for model in [model_a,model_b]:
        t=_severity_participant_contrib(oof,model,metric)
        for r in repeats:
            lookup[(model,r)]=t.loc[t["repeat"].eq(r)].set_index("participant_id").reindex(participants)["value"].to_numpy(float)
    rng=np.random.default_rng(seed); draws=np.empty(B,float)
    def collapse(v): return float(np.sqrt(np.mean(v))) if metric=="RMSE" else float(np.mean(v))
    for b in range(B):
        idx=rng.integers(0,len(participants),size=len(participants)); vals=[]
        for r in repeats:
            vals.append(collapse(lookup[(model_b,r)][idx])-collapse(lookup[(model_a,r)][idx]))
        draws[b]=np.mean(vals)
    obs=np.mean([collapse(lookup[(model_b,r)])-collapse(lookup[(model_a,r)]) for r in repeats])
    return float(obs),draws

bootstrap_rows=[]; bootstrap_draw_rows=[]
for model_b,model_a in [("M_A+Q","M_A"),("M_A-resQ","M_A")]:
    for metric in ["AUROC","Brier","CalibrationIntercept","CalibrationSlope"]:
        est,draws=bootstrap_contrast_diagnosis(dx_participant_repeat,model_b,model_a,metric,N_COMPLETION_BOOTSTRAPS,
            deterministic_seed("goal2_completion_boot",model_b,metric))
        bootstrap_rows.append({"task":"diagnosis","contrast":f"{model_b} minus {model_a}","metric":metric,
                               "estimate":est,"ci_low":float(np.quantile(draws,.025)),"ci_high":float(np.quantile(draws,.975)),
                               "bootstrap_replicates":len(draws)})
        bootstrap_draw_rows.extend({"task":"diagnosis","contrast":f"{model_b} minus {model_a}","metric":metric,"bootstrap":i+1,"delta":float(v)} for i,v in enumerate(draws))
    for metric in ["MAE","RMSE"]:
        est,draws=bootstrap_contrast_severity(sev_oof,model_b,model_a,metric,N_COMPLETION_BOOTSTRAPS,
            deterministic_seed("goal2_completion_boot",model_b,metric))
        bootstrap_rows.append({"task":"severity","contrast":f"{model_b} minus {model_a}","metric":metric,
                               "estimate":est,"ci_low":float(np.quantile(draws,.025)),"ci_high":float(np.quantile(draws,.975)),
                               "bootstrap_replicates":len(draws)})
        bootstrap_draw_rows.extend({"task":"severity","contrast":f"{model_b} minus {model_a}","metric":metric,"bootstrap":i+1,"delta":float(v)} for i,v in enumerate(draws))

authoritative_bootstrap=pd.DataFrame(bootstrap_rows)
atomic_write_csv(authoritative_bootstrap,TABLES/"goal2_metrics_authoritative.csv")
atomic_write_csv(pd.DataFrame(bootstrap_draw_rows),TABLES/"goal2_paired_bootstrap_draws_authoritative.csv")

# Final participant/pair predictions and shifts.
dx_final=(dx_participant_repeat.groupby(["model","participant_id"],as_index=False).agg(y=("y","first"),prediction=("prediction","mean")))
atomic_write_csv(dx_final,TABLES/"goal2_dx_participant_oof_authoritative.csv")
shift_rows=[]
for task,source,idcols in [("diagnosis",dx_final,["participant_id"]),("severity",sev_oof,["participant_id","logical_recording_id"] + (["assessment_date"] if "assessment_date" in sev_oof.columns else []))]:
    if task=="severity":
        source=(source.groupby(["model"]+idcols,as_index=False).agg(y=("y","first"),prediction=("prediction","mean")))
    base=source.loc[source["model"].eq("M_A"),idcols+["y","prediction"]].rename(columns={"prediction":"base_prediction"})
    for model in ["M_A+Q","M_A-resQ"]:
        comp=source.loc[source["model"].eq(model),idcols+["prediction"]].rename(columns={"prediction":"comparison_prediction"})
        merged=base.merge(comp,on=idcols,how="inner",validate="one_to_one")
        merged["task"]=task; merged["comparison"]=f"{model} minus M_A"
        merged["signed_shift"]=merged["comparison_prediction"]-merged["base_prediction"]
        merged["absolute_shift"]=merged["signed_shift"].abs()
        shift_rows.append(merged)
prediction_shifts=pd.concat(shift_rows,ignore_index=True)
atomic_write_csv(prediction_shifts,TABLES/"goal2_prediction_shifts_authoritative.csv")

display(authoritative_metrics)
display(authoritative_bootstrap)
print("AUTHORITATIVE METRICS + 2,000 PAIRED BOOTSTRAPS: PASS")


## 19. Recompute natural Q-dependent error with corrected `M_A-resQ`

In [ ]:
def collapse_oof_for_q_loss(
    oof,
    task,
):
    group_keys = [
        "model",
        "participant_id",
        "logical_recording_id",
        "y",
    ]

    if (
        task == "severity"
        and "assessment_date" in oof.columns
    ):
        group_keys.append(
            "assessment_date"
        )

    agg = {
        "prediction": "mean",
    }

    for q in CORE_Q + Q_SUPPORT:
        agg[q] = "mean"

    collapsed = (
        oof.groupby(
            group_keys,
            as_index=False,
        )
        .agg(agg)
    )

    if task == "diagnosis":
        collapsed["loss"] = (
            (
                collapsed["y"]
                - collapsed["prediction"]
            ) ** 2
        )
    else:
        collapsed["loss"] = np.abs(
            collapsed["y"]
            - collapsed["prediction"]
        )

    return collapsed

dx_q_loss = collapse_oof_for_q_loss(
    dx_oof,
    "diagnosis",
)

sev_q_loss = collapse_oof_for_q_loss(
    sev_oof,
    "severity",
)

goal2_oof_recording = pd.concat(
    [
        dx_q_loss.assign(
            task="diagnosis"
        ),
        sev_q_loss.assign(
            task="severity"
        ),
    ],
    ignore_index=True,
)

atomic_write_csv(
    goal2_oof_recording,
    TABLES / "goal2_oof_recording.csv",
)

print(
    "Final diagnosis recording/model rows:",
    len(dx_q_loss),
)
print(
    "Final severity pair/model rows:",
    len(sev_q_loss),
)

print("Q-LOSS OOF TABLE: PASS")


In [ ]:
Q_FAMILY = {
    feature: (
        q_registry.set_index("feature")
        .loc[feature, "family"]
    )
    for feature in CORE_Q
}

def bh_fdr(p_values):
    p = np.asarray(
        p_values,
        dtype=float,
    )

    q = np.full(
        len(p),
        np.nan,
        dtype=float,
    )

    valid = np.isfinite(p)

    if not valid.any():
        return q

    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]

    m = len(ranked)

    adjusted = (
        ranked
        * m
        / np.arange(
            1,
            m + 1,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.clip(
        adjusted,
        0,
        1,
    )

    restored = np.empty_like(
        adjusted
    )
    restored[order] = adjusted

    q[valid] = restored
    return q

def _linear_combination(
    result,
    coefficient_weights,
):
    names = list(
        result.params.index
    )

    vector = np.zeros(
        len(names),
        dtype=float,
    )

    for name, weight in coefficient_weights.items():
        if name not in names:
            raise KeyError(
                f"GEE coefficient not found: {name}"
            )
        vector[
            names.index(name)
        ] = weight

    estimate = float(
        vector
        @ result.params.to_numpy(float)
    )

    covariance = result.cov_params().to_numpy(float)

    variance = float(
        vector
        @ covariance
        @ vector
    )

    se = float(
        np.sqrt(
            max(
                variance,
                0.0,
            )
        )
    )

    z = (
        estimate / se
        if se > 0
        else np.nan
    )

    p_value = (
        2
        * stats.norm.sf(
            abs(z)
        )
        if np.isfinite(z)
        else np.nan
    )

    return (
        estimate,
        se,
        estimate - 1.96 * se,
        estimate + 1.96 * se,
        p_value,
    )

def gee_numeric_q(
    source,
    task,
    q_feature,
    cov_structure="independence",
):
    # One Q value per row identity; model creates stacked pseudo-observations.
    identity_keys = [
        "participant_id",
        "logical_recording_id",
    ]

    if (
        task == "severity"
        and "assessment_date" in source.columns
    ):
        identity_keys.append(
            "assessment_date"
        )

    q_base = (
        source[
            identity_keys + [q_feature]
        ]
        .drop_duplicates(
            identity_keys
        )
        .copy()
    )

    q_base[q_feature] = pd.to_numeric(
        q_base[q_feature],
        errors="coerce",
    )

    q_base = q_base.loc[
        np.isfinite(
            q_base[q_feature]
        )
    ].copy()

    if len(q_base) < 50:
        return pd.DataFrame()

    participant_mean = (
        q_base.groupby(
            "participant_id"
        )[q_feature]
        .mean()
    )

    q_base["q_between_raw"] = (
        q_base["participant_id"]
        .map(participant_mean)
    )

    q_base["q_within_raw"] = (
        q_base[q_feature]
        - q_base["q_between_raw"]
    )

    support_counts = (
        q_base.groupby(
            "participant_id"
        ).size()
    )
    within_eligible_ids = set(
        support_counts.loc[
            support_counts >= 2
        ].index.astype(str)
    )

    # Participants with a single supported recording have q_within=0
    # by construction and therefore do not identify the within-person
    # coefficient. Record the actual number of eligible clusters.
    n_within_eligible_participants = len(
        within_eligible_ids
    )

    q_iqr = float(
        np.quantile(
            q_base[q_feature],
            .75,
        )
        - np.quantile(
            q_base[q_feature],
            .25,
        )
    )

    if (
        not np.isfinite(q_iqr)
        or q_iqr <= 0
    ):
        return pd.DataFrame()

    q_base["q_between"] = (
        q_base["q_between_raw"]
        / q_iqr
    )

    q_base["q_within"] = (
        q_base["q_within_raw"]
        / q_iqr
    )

    merged = source.merge(
        q_base[
            identity_keys
            + [
                "q_between",
                "q_within",
            ]
        ],
        on=identity_keys,
        how="inner",
        validate="many_to_one",
    )

    merged["m_addq"] = (
        merged["model"].eq(
            "M_A+Q"
        ).astype(int)
    )

    merged["m_resq"] = (
        merged["model"].eq(
            "M_A-resQ"
        ).astype(int)
    )

    formula = (
        "loss ~ m_addq + m_resq + "
        "q_between + q_within + "
        "m_addq:q_between + "
        "m_resq:q_between + "
        "m_addq:q_within + "
        "m_resq:q_within"
    )

    cov_struct = (
        Independence()
        if cov_structure == "independence"
        else Exchangeable()
    )

    try:
        result = sm.GEE.from_formula(
            formula,
            groups="participant_id",
            data=merged,
            family=Gaussian(),
            cov_struct=cov_struct,
        ).fit()
    except Exception as exc:
        return pd.DataFrame([{
            "task": task,
            "q_feature": q_feature,
            "q_family": Q_FAMILY[q_feature],
            "cov_structure": cov_structure,
            "status": (
                f"failed:{type(exc).__name__}:{exc}"
            ),
        }])

    rows = []

    slope_weights = {
        "M_A": {
            "between": {
                "q_between": 1,
            },
            "within": {
                "q_within": 1,
            },
        },
        "M_A+Q": {
            "between": {
                "q_between": 1,
                "m_addq:q_between": 1,
            },
            "within": {
                "q_within": 1,
                "m_addq:q_within": 1,
            },
        },
        "M_A-resQ": {
            "between": {
                "q_between": 1,
                "m_resq:q_between": 1,
            },
            "within": {
                "q_within": 1,
                "m_resq:q_within": 1,
            },
        },
    }

    for model in MODELS:
        for component in [
            "between",
            "within",
        ]:
            (
                estimate,
                se,
                ci_low,
                ci_high,
                p_value,
            ) = _linear_combination(
                result,
                slope_weights[
                    model
                ][component],
            )

            rows.append({
                "task": task,
                "q_feature": q_feature,
                "q_family": Q_FAMILY[
                    q_feature
                ],
                "model": model,
                "component": component,
                "effect_per_1IQR_Q": estimate,
                "se": se,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "p_value": p_value,
                "q_iqr": q_iqr,
                "n_rows": len(merged),
                "n_participants": merged[
                    "participant_id"
                ].nunique(),
                "n_within_eligible_participants": (
                    n_within_eligible_participants
                ),
                "cov_structure": cov_structure,
                "status": "ok",
            })

    # Paired loss-change slopes for panels C/D.
    wide_index = list(
        dict.fromkeys(
            identity_keys
            + [
                "q_between",
                "q_within",
            ]
        )
    )

    wide = (
        merged.pivot_table(
            index=wide_index,
            columns="model",
            values="loss",
            aggfunc="first",
        )
        .reset_index()
    )

    for comparison in [
        "M_A+Q",
        "M_A-resQ",
    ]:
        wide["delta_loss"] = (
            wide[comparison]
            - wide["M_A"]
        )

        try:
            delta_result = (
                sm.GEE.from_formula(
                    (
                        "delta_loss ~ "
                        "q_between + q_within"
                    ),
                    groups="participant_id",
                    data=wide,
                    family=Gaussian(),
                    cov_struct=cov_struct,
                ).fit()
            )

            for component in [
                "between",
                "within",
            ]:
                term = (
                    "q_between"
                    if component == "between"
                    else "q_within"
                )

                (
                    estimate,
                    se,
                    ci_low,
                    ci_high,
                    p_value,
                ) = _linear_combination(
                    delta_result,
                    {term: 1},
                )

                rows.append({
                    "task": task,
                    "q_feature": q_feature,
                    "q_family": Q_FAMILY[
                        q_feature
                    ],
                    "model": (
                        f"{comparison}-minus-M_A-loss"
                    ),
                    "component": component,
                    "effect_per_1IQR_Q": estimate,
                    "se": se,
                    "ci_low": ci_low,
                    "ci_high": ci_high,
                    "p_value": p_value,
                    "q_iqr": q_iqr,
                    "n_rows": len(wide),
                    "n_participants": wide[
                        "participant_id"
                    ].nunique(),
                    "n_within_eligible_participants": (
                        n_within_eligible_participants
                    ),
                    "cov_structure": cov_structure,
                    "status": "ok",
                })

        except Exception as exc:
            rows.append({
                "task": task,
                "q_feature": q_feature,
                "q_family": Q_FAMILY[
                    q_feature
                ],
                "model": (
                    f"{comparison}-minus-M_A-loss"
                ),
                "component": "failed",
                "cov_structure": cov_structure,
                "status": (
                    f"failed:{type(exc).__name__}:{exc}"
                ),
            })

    return pd.DataFrame(rows)


def gee_support_indicator(
    source,
    task,
    support_feature,
    cov_structure="independence",
):
    identity_keys = [
        "participant_id",
        "logical_recording_id",
    ]
    if (
        task == "severity"
        and "assessment_date" in source.columns
    ):
        identity_keys.append("assessment_date")

    base = (
        source[
            identity_keys + [support_feature]
        ]
        .drop_duplicates(identity_keys)
        .copy()
    )

    base["support_binary"] = pd.to_numeric(
        base[support_feature],
        errors="coerce",
    )

    base = base.loc[
        base["support_binary"].isin([0, 1])
    ].copy()

    if (
        len(base) < 50
        or base["support_binary"].nunique() < 2
    ):
        return pd.DataFrame()

    merged = source.merge(
        base[
            identity_keys + ["support_binary"]
        ],
        on=identity_keys,
        how="inner",
        validate="many_to_one",
    )

    merged["m_addq"] = (
        merged["model"].eq("M_A+Q").astype(int)
    )
    merged["m_resq"] = (
        merged["model"].eq("M_A-resQ").astype(int)
    )

    cov_struct = (
        Independence()
        if cov_structure == "independence"
        else Exchangeable()
    )

    try:
        result = sm.GEE.from_formula(
            (
                "loss ~ m_addq + m_resq + support_binary + "
                "m_addq:support_binary + m_resq:support_binary"
            ),
            groups="participant_id",
            data=merged,
            family=Gaussian(),
            cov_struct=cov_struct,
        ).fit()
    except Exception as exc:
        return pd.DataFrame([{
            "task": task,
            "q_feature": support_feature,
            "q_family": "QADD",
            "model": "failed",
            "component": "support_binary",
            "cov_structure": cov_structure,
            "status": (
                f"failed:{type(exc).__name__}:{exc}"
            ),
        }])

    weights = {
        "M_A": {"support_binary": 1},
        "M_A+Q": {
            "support_binary": 1,
            "m_addq:support_binary": 1,
        },
        "M_A-resQ": {
            "support_binary": 1,
            "m_resq:support_binary": 1,
        },
    }

    rows = []

    for model in MODELS:
        (
            estimate,
            se,
            ci_low,
            ci_high,
            p_value,
        ) = _linear_combination(
            result,
            weights[model],
        )

        rows.append({
            "task": task,
            "q_feature": support_feature,
            "q_family": "QADD",
            "model": model,
            "component": "support_binary",
            "effect_per_1IQR_Q": np.nan,
            "effect_support_present_minus_absent": estimate,
            "se": se,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p_value": p_value,
            "q_iqr": np.nan,
            "n_rows": len(merged),
            "n_participants": merged[
                "participant_id"
            ].nunique(),
            "n_within_eligible_participants": np.nan,
            "cov_structure": cov_structure,
            "status": "ok",
        })

    return pd.DataFrame(rows)


association_tables = []

for task, source in [
    ("diagnosis", dx_q_loss),
    ("severity", sev_q_loss),
]:
    print(
        f"\nQ-loss GEE: {task}"
    )

    for q_feature in CORE_Q:
        print(
            " ",
            q_feature,
        )

        primary = gee_numeric_q(
            source,
            task,
            q_feature,
            cov_structure="independence",
        )

        if len(primary):
            association_tables.append(
                primary
            )

        sensitivity = gee_numeric_q(
            source,
            task,
            q_feature,
            cov_structure="exchangeable",
        )

        if len(sensitivity):
            association_tables.append(
                sensitivity
            )

    # QADD support indicators are prespecified separate binary tests.
    for support_feature in Q_SUPPORT:
        support_primary = gee_support_indicator(
            source,
            task,
            support_feature,
            cov_structure="independence",
        )
        if len(support_primary):
            association_tables.append(
                support_primary
            )

        support_sensitivity = gee_support_indicator(
            source,
            task,
            support_feature,
            cov_structure="exchangeable",
        )
        if len(support_sensitivity):
            association_tables.append(
                support_sensitivity
            )

q_error_associations = pd.concat(
    association_tables,
    ignore_index=True,
)

# BH-FDR only for successful primary-independence tests.
q_error_associations["q_value"] = np.nan

mask_primary = (
    q_error_associations[
        "cov_structure"
    ].eq("independence")
    & q_error_associations[
        "status"
    ].eq("ok")
)

for keys, idx in q_error_associations.loc[
    mask_primary
].groupby(
    [
        "task",
        "model",
        "component",
        "q_family",
    ]
).groups.items():
    q_error_associations.loc[
        idx,
        "q_value",
    ] = bh_fdr(
        q_error_associations.loc[
            idx,
            "p_value",
        ]
    )

atomic_write_csv(
    q_error_associations,
    TABLES / "goal2_q_error_associations.csv",
    required_columns=[
        "task",
        "q_feature",
        "q_family",
        "model",
        "component",
        "effect_per_1IQR_Q",
        "ci_low",
        "ci_high",
        "p_value",
        "cov_structure",
    ],
)

display(
    q_error_associations.loc[
        mask_primary
    ].sort_values(
        [
            "task",
            "component",
            "q_family",
            "q_value",
        ]
    )
)

print("NATURAL Q-DEPENDENT ERROR ANALYSIS: PASS")


## 20. Low-cost prespecified sensitivities: median diagnosis aggregation, A-only, first-recording diagnosis, and Extended-Q


In [ ]:

# 20a. Median instead of mean participant aggregation: no refit.
dx_median_pr = participant_dx_by_repeat_local(dx_oof, "median")
dx_median_metrics = diagnosis_repeat_metrics_local(dx_median_pr)
median_summary=(dx_median_metrics.groupby("model",as_index=False).agg(AUROC=("AUROC","mean"),Brier=("Brier","mean")))
median_summary["sensitivity"]="median participant aggregation"
atomic_write_csv(median_summary,TABLES/"goal2_sensitivity_median_aggregation.csv")

# 20b. A-only same primary rows.
A_ONLY_SPEC = make_spec("A-only", include_age=False)
aonly_dx,aonly_dx_manifest=run_ridge_spec_cv(dx,"diagnosis",A_ONLY_SPEC,label="sensitivity_A_only")
aonly_sev,aonly_sev_manifest=run_ridge_spec_cv(sev,"severity",A_ONLY_SPEC,label="sensitivity_A_only")

# 20c. Frozen first chronological diagnosis recording per participant, restricted to the primary age-complete cohort.
first_index_path = PROCESSED / "goal1_diagnosis_index.csv"
if not first_index_path.exists():
    raise FileNotFoundError("Frozen Goal 1 diagnosis index is required for first-recording sensitivity.")
first_index=safe_read_csv(first_index_path,required_columns=["participant_id","logical_recording_id"])
first_keys=first_index[["participant_id","logical_recording_id"]].drop_duplicates()
dx_first=dx.merge(first_keys,on=["participant_id","logical_recording_id"],how="inner",validate="many_to_one")
if dx_first["participant_id"].nunique()!=dx["participant_id"].nunique():
    raise RuntimeError("First-recording sensitivity lost an age-complete diagnosis participant.")
first_outputs={}
for model,spec in PRIMARY_SPECS.items():
    out,man=run_ridge_spec_cv(dx_first,"diagnosis",spec,label=f"sensitivity_first_recording_{model}")
    first_outputs[model]=out

# 20d. Extended-Q. Preserve Primary-A; expand only Q according to the frozen Extended-Q sensitivity contract.
EXT_Q_AQ_SPEC=make_spec(
    "M_A+ExtendedQ",q_features=EXTENDED_Q,q_support_sources=EXTENDED_SUPPORT_SOURCES,
    q_binary_cols=[PERSISTENCE_CENSOR]
)
EXT_Q_RES_SPEC=make_spec(
    "M_A-resExtendedQ",q_features=EXTENDED_Q,q_support_sources=EXTENDED_SUPPORT_SOURCES,
    q_binary_cols=[PERSISTENCE_CENSOR],residualize=True
)
ext_dx_aq,ext_dx_aq_manifest=run_ridge_spec_cv(dx,"diagnosis",EXT_Q_AQ_SPEC,label="sensitivity_extendedQ_AplusQ")
ext_dx_res,ext_dx_res_manifest=run_ridge_spec_cv(dx,"diagnosis",EXT_Q_RES_SPEC,label="sensitivity_extendedQ_resQ")
ext_sev_aq,ext_sev_aq_manifest=run_ridge_spec_cv(sev,"severity",EXT_Q_AQ_SPEC,label="sensitivity_extendedQ_AplusQ")
ext_sev_res,ext_sev_res_manifest=run_ridge_spec_cv(sev,"severity",EXT_Q_RES_SPEC,label="sensitivity_extendedQ_resQ")

print("LOW-COST / RIDGE SENSITIVITY REFITS: COMPLETE")
print("A-only diagnosis participants:",aonly_dx["participant_id"].nunique())
print("First-recording diagnosis participants:",dx_first["participant_id"].nunique())


## 21. Q-family add-one and leave-one-out ablations

These are secondary localization analyses. Add-one models compare `Age+A+one Q family` with `M_A`; leave-one-out models compare `Age+A+Core-Q minus one family` with full `M_A+Q`. No acoustic features are reselected.


In [ ]:

family_outputs={}
family_specs={}
for family,features in Q_FAMILIES.items():
    add_support=QADD if family=="QADD" else []
    family_specs[("add_one",family)]=make_spec(
        f"A+{family}",q_features=features,q_support_sources=add_support
    )
    leave_features=[f for f in CORE_Q if f not in features]
    leave_support=QADD if family!="QADD" else []
    family_specs[("leave_one_out",family)]=make_spec(
        f"A+Q_without_{family}",q_features=leave_features,q_support_sources=leave_support
    )

for task,frame in [("diagnosis",dx),("severity",sev)]:
    for (kind,family),spec in family_specs.items():
        out,man=run_ridge_spec_cv(frame,task,spec,label=f"family_{kind}_{family}")
        family_outputs[(task,kind,family)]=out

print("Q-FAMILY ADD-ONE / LEAVE-ONE-OUT REFITS: COMPLETE")


## 22. Constrained HGB model-form sensitivity with group-safe early stopping

Scikit-learn's default internal early-stopping split is not used because repeated recordings from one participant could cross that internal boundary. Instead, iteration count is selected from staged predictions on the same participant-grouped inner folds used for tuning. This preserves the frozen HGB constraints while maintaining participant independence.


In [ ]:

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor


def make_hgb(task, params, max_iter, seed):
    common=dict(
        max_depth=int(params["max_depth"]),
        learning_rate=float(params["learning_rate"]),
        min_samples_leaf=int(params["min_samples_leaf"]),
        l2_regularization=float(params["l2_regularization"]),
        max_iter=int(max_iter),
        early_stopping=False,
        random_state=int(seed % (2**31-1)),
    )
    return HistGradientBoostingClassifier(**common) if task=="diagnosis" else HistGradientBoostingRegressor(**common)


def staged_validation_losses(model, Xv, yv, wv, task):
    losses=[]
    iterator=model.staged_predict_proba(Xv) if task=="diagnosis" else model.staged_predict(Xv)
    for stage in iterator:
        pred=stage[:,1] if task=="diagnosis" else stage
        losses.append(validation_loss(yv,pred,wv,task))
    return np.asarray(losses,float)


def tune_hgb_spec(train, task, seed, spec, residualizer_alpha=None):
    folds=make_inner_participant_folds(train,task,seed)
    bundles=[]
    for fold_idx,(tr_ids,va_ids) in enumerate(folds,start=1):
        tr=train.loc[train["participant_id"].isin(tr_ids)].copy(); va=train.loc[train["participant_id"].isin(va_ids)].copy()
        tr["row_weight"]=participant_weights(tr); va["row_weight"]=participant_weights(va)
        Xtr,Xv,_=fit_design_spec(tr,va,spec,reference_ids=set(tr["participant_id"].astype(str)),residualizer_alpha=residualizer_alpha)
        bundles.append((tr,va,Xtr,Xv))
    candidates=[]
    for param_idx,params in enumerate(HGB_GRID):
        fold_losses=[]
        for fold_idx,(tr,va,Xtr,Xv) in enumerate(bundles,start=1):
            model=make_hgb(task,params,HGB_MAX_ITER,deterministic_seed("hgb_inner",seed,param_idx,fold_idx))
            model.fit(Xtr,tr["y"].to_numpy(float if task=="severity" else int),sample_weight=tr["row_weight"].to_numpy(float))
            loss=staged_validation_losses(model,Xv,va["y"].to_numpy(float),va["row_weight"].to_numpy(float),task)
            if len(loss)!=HGB_MAX_ITER:
                raise RuntimeError("HGB staged prediction length mismatch.")
            fold_losses.append(loss)
        mean_curve=np.mean(np.vstack(fold_losses),axis=0)
        best_iter=int(np.argmin(mean_curve))+1
        candidates.append({**params,"max_iter":best_iter,"mean_inner_loss":float(mean_curve[best_iter-1])})
    cand=pd.DataFrame(candidates).sort_values(["mean_inner_loss","max_depth","learning_rate","min_samples_leaf","l2_regularization","max_iter"],kind="stable")
    return cand.iloc[0].to_dict(),cand


def run_hgb_spec_cv(frame,task,spec,*,label):
    out_dir=CHECKPOINTS/_slug(label)/task; out_dir.mkdir(parents=True,exist_ok=True)
    folds=[]; manifests=[]
    for repeat in range(1,OUTER_REPEATS+1):
        print(f"{label} | {task} | HGB repeat {repeat}/{OUTER_REPEATS}")
        for outer_fold in range(1,OUTER_FOLDS+1):
            fp=out_dir/f"r{repeat:02d}_f{outer_fold:02d}.csv"; mp=out_dir/f"r{repeat:02d}_f{outer_fold:02d}.json"
            if RESUME_IF_VALID and fp.exists() and mp.exists() and fp.stat().st_size>0:
                folds.append(safe_read_csv(fp,required_columns=["participant_id","logical_recording_id","repeat","outer_fold","model","y","prediction","row_weight"]))
                manifests.append(json.loads(mp.read_text(encoding="utf-8"))); continue
            test_ids=set(split_manifest.loc[split_manifest["repeat"].eq(repeat)&split_manifest["outer_fold"].eq(outer_fold),"participant_id"].astype(str)) & set(frame["participant_id"].astype(str))
            train=frame.loc[~frame["participant_id"].isin(test_ids)].copy(); test=frame.loc[frame["participant_id"].isin(test_ids)].copy()
            train["row_weight"]=participant_weights(train); test["row_weight"]=participant_weights(test)
            seed=deterministic_seed("goal2_hgb",label,task,repeat,outer_fold)
            resid_alpha=None
            if spec["residualize"]:
                resid_alpha,_=select_residualizer_alpha(train,task,seed+7,spec)
            selected,candidates=tune_hgb_spec(train,task,seed+11,spec,residualizer_alpha=resid_alpha)
            params={k:selected[k] for k in ["max_depth","learning_rate","min_samples_leaf","l2_regularization"]}
            max_iter=int(selected["max_iter"])
            Xtr,Xte,_=fit_design_spec(train,test,spec,reference_ids=set(train["participant_id"].astype(str)),residualizer_alpha=resid_alpha)
            model=make_hgb(task,params,max_iter,seed+19)
            model.fit(Xtr,train["y"].to_numpy(float if task=="severity" else int),sample_weight=train["row_weight"].to_numpy(float))
            pred=model.predict_proba(Xte)[:,1] if task=="diagnosis" else model.predict(Xte)
            cols=["participant_id","logical_recording_id","y","row_weight"]
            if task=="severity":
                for c in ["assessment_date","abs_delta_days"]:
                    if c in test.columns: cols.append(c)
            result=test[cols].copy(); result["repeat"]=repeat; result["outer_fold"]=outer_fold; result["model"]=f"HGB_{spec['name']}"; result["prediction"]=pred
            result["loss"]=(result["y"]-result["prediction"])**2 if task=="diagnosis" else np.abs(result["y"]-result["prediction"])
            atomic_write_csv(result,fp)
            meta={"task":task,"repeat":repeat,"outer_fold":outer_fold,"model":f"HGB_{spec['name']}",
                  "selected_params":{**params,"max_iter":max_iter},"selected_residualizer_alpha":resid_alpha}
            atomic_write_json(meta,mp); folds.append(result); manifests.append(meta)
    return pd.concat(folds,ignore_index=True),pd.DataFrame(manifests)

hgb_outputs={}; hgb_manifests=[]
if RUN_HGB:
    for task,frame in [("diagnosis",dx),("severity",sev)]:
        for model,spec in PRIMARY_SPECS.items():
            oof,man=run_hgb_spec_cv(frame,task,spec,label=f"HGB_{model}")
            hgb_outputs[(task,model)]=oof; hgb_manifests.append(man)
    atomic_write_csv(pd.concat(hgb_manifests,ignore_index=True),TABLES/"goal2_hgb_fold_manifest.csv")
    print("GROUP-SAFE HGB SENSITIVITY: COMPLETE")
else:
    print("HGB SKIPPED BY CONFIGURATION — Goal 2 cannot receive the complete seal until RUN_HGB=True.")


## 23. Robustness of Q-loss inference: participant-cluster bootstrap, rank sensitivity, severity log1p(error), and overlap diagnostics

In [ ]:

# The primary GEE table above already includes independence (primary) and exchangeable (sensitivity).
# Here we add two distributional checks and a participant-cluster bootstrap for numeric Q slopes.

def q_component_frame(source, task, model, q_feature):
    g=source.loc[source["model"].eq(model)].copy()
    id_keys=["participant_id","logical_recording_id"]
    if task=="severity" and "assessment_date" in g.columns: id_keys.append("assessment_date")
    base=g[id_keys+[q_feature,"loss"]].drop_duplicates(id_keys).copy()
    base=base.loc[base[q_feature].notna() & base["loss"].notna()].copy()
    counts=base.groupby("participant_id")[q_feature].transform("count")
    means=base.groupby("participant_id")[q_feature].transform("mean")
    base["q_between_raw"]=means
    base["q_within_raw"]=base[q_feature]-means
    iqr=float(np.nanquantile(base[q_feature],.75)-np.nanquantile(base[q_feature],.25))
    if not np.isfinite(iqr) or iqr<=0: return None
    base["q_between"]=base["q_between_raw"]/iqr
    base["q_within"]=base["q_within_raw"]/iqr
    base["within_supported"]=counts.ge(2)
    return base


def bootstrap_q_slopes(base,B,seed):
    participants=np.array(sorted(base["participant_id"].astype(str).unique()))
    by={pid:base.loc[base["participant_id"].astype(str).eq(pid)].copy() for pid in participants}
    rng=np.random.default_rng(seed); bcoef=np.full((B,2),np.nan)
    for b in range(B):
        sampled=rng.choice(participants,size=len(participants),replace=True)
        pieces=[]
        for draw,pid in enumerate(sampled):
            x=by[pid].copy(); x["__cluster"]=draw; pieces.append(x)
        boot=pd.concat(pieces,ignore_index=True)
        # Joint linear sensitivity: loss ~ between + within. Within rows lacking repeated support contribute q_within=0.
        qwithin=np.where(boot["within_supported"].to_numpy(bool),boot["q_within"].to_numpy(float),0.0)
        X=np.column_stack([np.ones(len(boot)),boot["q_between"].to_numpy(float),qwithin])
        y=boot["loss"].to_numpy(float)
        try:
            coef=np.linalg.lstsq(X,y,rcond=None)[0]
            bcoef[b,:]=coef[1:3]
        except Exception:
            pass
    return bcoef

robust_rows=[]
for task,source in [("diagnosis",dx_q_loss),("severity",sev_q_loss)]:
    for model in MODELS:
        for q in CORE_Q:
            base=q_component_frame(source,task,model,q)
            if base is None or base["participant_id"].nunique()<20: continue
            bcoef=bootstrap_q_slopes(base,N_COMPLETION_BOOTSTRAPS,deterministic_seed("q_slope_boot",task,model,q))
            for j,component in enumerate(["between","within"]):
                vals=bcoef[:,j]; vals=vals[np.isfinite(vals)]
                robust_rows.append({"task":task,"model":model,"q_feature":q,"sensitivity":"participant_cluster_bootstrap",
                                    "component":component,"estimate":float(np.median(vals)),
                                    "ci_low":float(np.quantile(vals,.025)),"ci_high":float(np.quantile(vals,.975)),"n_bootstrap":len(vals)})
            # Rank-based cluster-robust GEE per model.
            rb=base.copy()
            rb["loss_rank"]=stats.rankdata(rb["loss"].to_numpy(float),method="average")/len(rb)
            rb["qb_rank"]=stats.rankdata(rb["q_between"].to_numpy(float),method="average")/len(rb)
            rb["qw_rank"]=stats.rankdata(rb["q_within"].to_numpy(float),method="average")/len(rb)
            X=sm.add_constant(rb[["qb_rank","qw_rank"]].to_numpy(float))
            try:
                fit=sm.GEE(rb["loss_rank"].to_numpy(float),X,groups=rb["participant_id"].astype(str).to_numpy(),
                           family=Gaussian(),cov_struct=Independence()).fit()
                for j,component in [(1,"between"),(2,"within")]:
                    robust_rows.append({"task":task,"model":model,"q_feature":q,"sensitivity":"rank_GEE",
                                        "component":component,"estimate":float(fit.params[j]),
                                        "ci_low":float(fit.conf_int()[j,0]),"ci_high":float(fit.conf_int()[j,1]),
                                        "p_value":float(fit.pvalues[j])})
            except Exception:
                pass

# Severity log1p(error) GEE uses the same exact primary machinery after transforming only the loss.
sev_log=sev_q_loss.copy(); sev_log["loss"]=np.log1p(sev_log["loss"].to_numpy(float))
for q in CORE_Q:
    try:
        t=gee_numeric_q(sev_log,"severity",q,cov_structure="independence")
        t["sensitivity"]="severity_log1p_error_GEE"; robust_rows.extend(t.to_dict("records"))
    except Exception as exc:
        robust_rows.append({"task":"severity","q_feature":q,"sensitivity":"severity_log1p_error_GEE","status":f"failed:{type(exc).__name__}"})

q_loss_sensitivity=pd.DataFrame(robust_rows)
atomic_write_csv(q_loss_sensitivity,TABLES/"goal2_q_loss_sensitivity.csv")

# Outcome-aware overlap diagnostics are descriptive only. We do not invent a post-hoc universal cut point.
# Central 5–95% participant-mean class intervals are reported so the conditional overlap sensitivity can be judged transparently.
q_once=dx_q_loss.loc[dx_q_loss["model"].eq("M_A")].copy()
participant_q=(q_once.groupby(["participant_id","y"],as_index=False)[CORE_Q].mean(numeric_only=True))
overlap_rows=[]
for q in CORE_Q:
    c0=participant_q.loc[participant_q["y"].eq(0),q].dropna().to_numpy(float)
    c1=participant_q.loc[participant_q["y"].eq(1),q].dropna().to_numpy(float)
    if len(c0)<10 or len(c1)<10: continue
    lo0,hi0=np.quantile(c0,[.05,.95]); lo1,hi1=np.quantile(c1,[.05,.95])
    lo=max(lo0,lo1); hi=min(hi0,hi1)
    overlap_rows.append({"q_feature":q,"control_q05":lo0,"control_q95":hi0,"als_q05":lo1,"als_q95":hi1,
                         "common_low":lo,"common_high":hi,"common_interval_nonempty":bool(lo<hi),
                         "control_fraction_in_common":float(np.mean((c0>=lo)&(c0<=hi))) if lo<hi else 0.0,
                         "als_fraction_in_common":float(np.mean((c1>=lo)&(c1<=hi))) if lo<hi else 0.0})
overlap_diagnostics=pd.DataFrame(overlap_rows)
atomic_write_csv(overlap_diagnostics,TABLES/"goal2_overlap_diagnostics.csv")
display(overlap_diagnostics)
print("Q-LOSS ROBUSTNESS + OVERLAP DIAGNOSTICS: COMPLETE")


## 24. Score every sensitivity and compute family-ablation inference

In [ ]:

def score_single_oof(oof,task,model_label=None):
    x=oof.copy()
    if model_label is not None: x["model"]=model_label
    if task=="diagnosis":
        pr=participant_dx_by_repeat_local(x,"mean"); m=diagnosis_repeat_metrics_local(pr)
        return {"AUROC":float(m["AUROC"].mean()),"Brier":float(m["Brier"].mean()),"n_participants":int(pr["participant_id"].nunique())}
    m=severity_repeat_metrics_local(x)
    return {"MAE":float(m["MAE"].mean()),"RMSE":float(m["RMSE"].mean()),"n_participants":int(x["participant_id"].nunique())}

sens_rows=[]
# Authoritative primary anchors.
for task,source in [("diagnosis",dx_oof),("severity",sev_oof)]:
    for model in MODELS:
        local=source.loc[source["model"].eq(model)].copy()
        sens_rows.append({"task":task,"sensitivity":"authoritative primary","model":model,**score_single_oof(local,task,model)})
# A-only.
sens_rows.append({"task":"diagnosis","sensitivity":"A-only vs Age+A","model":"A-only",**score_single_oof(aonly_dx,"diagnosis","A-only")})
sens_rows.append({"task":"severity","sensitivity":"A-only vs Age+A","model":"A-only",**score_single_oof(aonly_sev,"severity","A-only")})
# First recording.
for model,out in first_outputs.items(): sens_rows.append({"task":"diagnosis","sensitivity":"first recording only","model":model,**score_single_oof(out,"diagnosis",model)})
# Median agg.
for row in median_summary.to_dict("records"): sens_rows.append({"task":"diagnosis","sensitivity":"median participant aggregation","model":row["model"],"AUROC":row["AUROC"],"Brier":row["Brier"],"n_participants":dx["participant_id"].nunique()})
# Extended Q.
for task,aout,rout in [("diagnosis",ext_dx_aq,ext_dx_res),("severity",ext_sev_aq,ext_sev_res)]:
    sens_rows.append({"task":task,"sensitivity":"Extended-Q","model":"M_A+ExtendedQ",**score_single_oof(aout,task,"M_A+ExtendedQ")})
    sens_rows.append({"task":task,"sensitivity":"Extended-Q","model":"M_A-resExtendedQ",**score_single_oof(rout,task,"M_A-resExtendedQ")})
# HGB.
if RUN_HGB:
    for (task,model),out in hgb_outputs.items(): sens_rows.append({"task":task,"sensitivity":"HGB model-form","model":f"HGB_{model}",**score_single_oof(out,task,f"HGB_{model}")})

sensitivity_summary=pd.DataFrame(sens_rows)
atomic_write_csv(sensitivity_summary,TABLES/"goal2_sensitivity_summary.csv")

# Family ablations: performance + cluster-robust paired-loss test; BH within task x ablation type.
family_rows=[]
for task,primary_source in [("diagnosis",dx_oof),("severity",sev_oof)]:
    ref_base=primary_source.loc[primary_source["model"].eq("M_A")].copy()
    ref_full=primary_source.loc[primary_source["model"].eq("M_A+Q")].copy()
    for (t, kind, family), out in family_outputs.items():
        if t != task:
            continue
        reference=ref_base if kind=="add_one" else ref_full
        metric="AUROC" if task=="diagnosis" else "MAE"
        # Bootstrap primary performance contrast.
        if task=="diagnosis":
            pr_b=participant_dx_by_repeat_local(out,"mean"); pr_a=participant_dx_by_repeat_local(reference,"mean")
            # Rename both to stable labels and concatenate for paired bootstrap helper.
            pr_b["model"]="B"; pr_a["model"]="A"; pr=pd.concat([pr_a,pr_b],ignore_index=True)
            est,draws=bootstrap_contrast_diagnosis(pr,"B","A","AUROC",N_COMPLETION_BOOTSTRAPS,deterministic_seed("fam_boot",task,kind,family))
        else:
            b=out.copy(); a=reference.copy(); b["model"]="B"; a["model"]="A"; both=pd.concat([a,b],ignore_index=True)
            est,draws=bootstrap_contrast_severity(both,"B","A","MAE",N_COMPLETION_BOOTSTRAPS,deterministic_seed("fam_boot",task,kind,family))
        # Paired loss GEE intercept for inferential p-value.
        idkeys=["participant_id","logical_recording_id","repeat"]
        if task=="severity" and "assessment_date" in out.columns and "assessment_date" in reference.columns: idkeys.append("assessment_date")
        bo=out[idkeys+["loss"]].rename(columns={"loss":"loss_b"})
        aa=reference[idkeys+["loss"]].rename(columns={"loss":"loss_a"})
        pair=bo.merge(aa,on=idkeys,how="inner",validate="one_to_one")
        pair["delta_loss"]=pair["loss_b"]-pair["loss_a"]
        try:
            fit=sm.GEE(pair["delta_loss"].to_numpy(float),np.ones((len(pair),1)),groups=pair["participant_id"].astype(str).to_numpy(),family=Gaussian(),cov_struct=Independence()).fit()
            p=float(fit.pvalues[0]); mean_loss=float(fit.params[0])
        except Exception:
            p=np.nan; mean_loss=float(pair["delta_loss"].mean())
        family_rows.append({"task":task,"ablation_type":kind,"q_family":family,"metric":metric,
                            "performance_delta":est,"ci_low":float(np.quantile(draws,.025)),"ci_high":float(np.quantile(draws,.975)),
                            "mean_paired_loss_delta":mean_loss,"p_value":p})
family_ablations=pd.DataFrame(family_rows)
family_ablations["q_value"]=np.nan
for keys,idx in family_ablations.groupby(["task","ablation_type"]).groups.items():
    family_ablations.loc[idx,"q_value"]=bh_fdr(family_ablations.loc[idx,"p_value"].to_numpy(float))
atomic_write_csv(family_ablations,TABLES/"goal2_family_ablations.csv")

display(sensitivity_summary)
display(family_ablations)
print("SENSITIVITY SCORING + FAMILY ABLATION FDR: PASS")


## 25. Ridge coefficient stability across outer fits

Descriptive only. These are standardized penalized coefficients; **no coefficient p-values** are computed.


In [ ]:

# Reconstruct outer fits from selected hyperparameters. M_A/M_A+Q use Notebook 20's frozen selections;
# corrected M_A-resQ uses the completion manifest.
coef_rows=[]
corrected_lookup={(r.task,int(r.repeat),int(r.outer_fold)):r for r in corrected_manifest.itertuples(index=False)}
for task,frame in [("diagnosis",dx),("severity",sev)]:
    for repeat in range(1,OUTER_REPEATS+1):
        for outer_fold in range(1,OUTER_FOLDS+1):
            test_ids=set(split_manifest.loc[split_manifest["repeat"].eq(repeat)&split_manifest["outer_fold"].eq(outer_fold),"participant_id"].astype(str)) & set(frame["participant_id"].astype(str))
            train=frame.loc[~frame["participant_id"].isin(test_ids)].copy(); test=frame.loc[frame["participant_id"].isin(test_ids)].copy()
            train["row_weight"]=participant_weights(train); test["row_weight"]=participant_weights(test)
            for model_name in MODELS:
                spec=PRIMARY_SPECS[model_name]
                if model_name=="M_A-resQ":
                    m=corrected_lookup[(task,repeat,outer_fold)]
                    resid_alpha=float(m.selected_residualizer_alpha)
                    param=float(m.selected_hyperparameter)
                else:
                    resid_alpha=None
                    row=primary_fold_manifest.loc[
                        primary_fold_manifest["task"].eq(task)&primary_fold_manifest["repeat"].eq(repeat)&
                        primary_fold_manifest["outer_fold"].eq(outer_fold)&primary_fold_manifest["model"].eq(model_name)
                    ]
                    if len(row)!=1: raise RuntimeError("Primary fold hyperparameter lookup failed.")
                    param=float(row["selected_hyperparameter"].iloc[0])
                Xtr,Xte,meta=fit_design_spec(train,test,spec,reference_ids=set(train["participant_id"].astype(str)),residualizer_alpha=resid_alpha)
                mdl=fit_outcome_model(Xtr,train["y"].to_numpy(float),train["row_weight"].to_numpy(float),task,param)
                co=np.asarray(mdl.coef_).reshape(-1)
                if len(co)!=len(meta["feature_names"]): raise RuntimeError("Coefficient/name dimension mismatch.")
                coef_rows.extend({"task":task,"repeat":repeat,"outer_fold":outer_fold,"model":model_name,"feature":name,"coefficient":float(value)} for name,value in zip(meta["feature_names"],co))
coef_table=pd.DataFrame(coef_rows)
coef_summary=(coef_table.groupby(["task","model","feature"],as_index=False).agg(
    mean_coefficient=("coefficient","mean"),median_coefficient=("coefficient","median"),
    sd_coefficient=("coefficient","std"),median_abs_coefficient=("coefficient",lambda s:float(np.median(np.abs(s)))),
    positive_fraction=("coefficient",lambda s:float(np.mean(np.asarray(s)>0))),n_outer_fits=("coefficient","size")
))
coef_summary["sign_consistency"]=np.maximum(coef_summary["positive_fraction"],1-coef_summary["positive_fraction"])
atomic_write_csv(coef_table,TABLES/"goal2_ridge_coefficients_outer_fits.csv")
atomic_write_csv(coef_summary,TABLES/"goal2_ridge_coefficient_stability.csv")
print("RIDGE COEFFICIENT STABILITY: PASS")


## 26. Publication-ready supplementary Goal 2 completion figure

Figure S6 summarizes whether the Goal 2 story depends on alternative representation/row/model choices and localizes Q-family contributions.


In [ ]:

# Sophisticated muted palette consistent with the Goal 1/Goal 2 visual system.
PALETTE={
    "M_A":"#203D5B","M_A+Q":"#167C80","M_A-resQ":"#A65345",
    "QADD":"#3D78A8","QGAIN":"#C67645","QREV":"#52866B","QCHAN":"#846A98",
    "neutral":"#6B7280","light":"#D9DEE5",
}
MM_TO_IN=1/25.4
mpl.rcParams.update({
    "font.family":"Arial","font.size":7,"axes.labelsize":7,"xtick.labelsize":6,"ytick.labelsize":6,
    "legend.fontsize":6,"axes.linewidth":.7,"xtick.major.width":.6,"ytick.major.width":.6,
    "pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none","figure.facecolor":"white","axes.facecolor":"white",
})

def clean_ax(ax):
    ax.grid(False); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False); ax.tick_params(direction="out")

def plabel(ax,s): ax.text(-.12,1.04,s,transform=ax.transAxes,fontsize=8,fontweight="bold",va="bottom")

fig,axes=plt.subplots(2,2,figsize=(183*MM_TO_IN,145*MM_TO_IN),constrained_layout=True)
# A diagnosis sensitivity point estimates.
ax=axes[0,0]
d=sensitivity_summary.loc[sensitivity_summary["task"].eq("diagnosis") & sensitivity_summary["AUROC"].notna()].copy()
labels=(d["sensitivity"]+" | "+d["model"]).tolist(); y=np.arange(len(d)); x=d["AUROC"].to_numpy(float)
ax.scatter(x,y,s=18,c=[PALETTE.get(m.replace("HGB_",""),PALETTE["neutral"]) for m in d["model"]],zorder=3)
ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis(); ax.set_xlabel("Participant-level AUROC"); clean_ax(ax); plabel(ax,"a")
# B severity sensitivity.
ax=axes[0,1]
s=sensitivity_summary.loc[sensitivity_summary["task"].eq("severity") & sensitivity_summary["MAE"].notna()].copy()
labels=(s["sensitivity"]+" | "+s["model"]).tolist(); y=np.arange(len(s)); x=s["MAE"].to_numpy(float)
ax.scatter(x,y,s=18,c=[PALETTE.get(m.replace("HGB_",""),PALETTE["neutral"]) for m in s["model"]],zorder=3)
ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis(); ax.set_xlabel("Participant-weighted MAE (lower is better)"); clean_ax(ax); plabel(ax,"b")
# C diagnosis Q-family localization: add-one and leave-one-out on one common AUROC scale.
ax=axes[1,0]
order=["QADD","QGAIN","QREV","QCHAN"]
f=family_ablations.loc[family_ablations["task"].eq("diagnosis")].copy()
for kind,offset,marker,label in [
    ("add_one",-.09,"o","Add one family vs M_A"),
    ("leave_one_out",.09,"s","Leave one family out vs full M_A+Q"),
]:
    z=(f.loc[f["ablation_type"].eq(kind)].set_index("q_family").reindex(order))
    yy=np.arange(len(order))+offset
    vals=z["performance_delta"].to_numpy(float)
    lo=z["ci_low"].to_numpy(float); hi=z["ci_high"].to_numpy(float)
    for j,fam in enumerate(order):
        ax.errorbar(
            vals[j],yy[j],
            xerr=np.array([[max(0.0,vals[j]-lo[j])],[max(0.0,hi[j]-vals[j])]]),
            fmt=marker,capsize=2,linewidth=.8,markersize=3.5,
            color=PALETTE[fam],
            label=label if j==0 else None,
        )
ax.axvline(0,color=PALETTE["neutral"],linestyle="--",linewidth=.7)
ax.set_yticks(np.arange(4)); ax.set_yticklabels(order); ax.invert_yaxis()
ax.set_xlabel("Diagnosis performance change, ΔAUROC")
ax.legend(frameon=False); clean_ax(ax); plabel(ax,"c")

# D severity Q-family localization on the MAE scale only.
ax=axes[1,1]
f=family_ablations.loc[family_ablations["task"].eq("severity")].copy()
for kind,offset,marker,label in [
    ("add_one",-.09,"o","Add one family vs M_A"),
    ("leave_one_out",.09,"s","Leave one family out vs full M_A+Q"),
]:
    z=(f.loc[f["ablation_type"].eq(kind)].set_index("q_family").reindex(order))
    yy=np.arange(len(order))+offset
    vals=z["performance_delta"].to_numpy(float)
    lo=z["ci_low"].to_numpy(float); hi=z["ci_high"].to_numpy(float)
    for j,fam in enumerate(order):
        ax.errorbar(
            vals[j],yy[j],
            xerr=np.array([[max(0.0,vals[j]-lo[j])],[max(0.0,hi[j]-vals[j])]]),
            fmt=marker,capsize=2,linewidth=.8,markersize=3.5,
            color=PALETTE[fam],
            label=label if j==0 else None,
        )
ax.axvline(0,color=PALETTE["neutral"],linestyle="--",linewidth=.7)
ax.set_yticks(np.arange(4)); ax.set_yticklabels(order); ax.invert_yaxis()
ax.set_xlabel("Severity performance change, ΔMAE\n(negative = lower error / better)")
ax.legend(frameon=False); clean_ax(ax); plabel(ax,"d")

stem="FigureS6_Goal2_completion_sensitivities"
fig.savefig(FIGURES/f"{stem}.pdf",bbox_inches="tight")
fig.savefig(FIGURES/f"{stem}.svg",bbox_inches="tight")
fig.savefig(FIGURES/f"{stem}.png",dpi=600,bbox_inches="tight")
plt.show()
print("FIGURE S6: WRITTEN")


## 27. Regenerate the authoritative main Goal 2 figures and native calibration supplement

Because the protocol-concordant residualizer may change `M_A-resQ`, Figures 3–4 and the calibration supplement are regenerated from the completion tables. These supersede Notebook 20's earlier residualized-model panels.


In [ ]:

# Human-readable feature labels kept short to prevent crowding.
Q_LABELS={
    "qadd_pause_ac_level_dbfs_median":"Pause level",
    "qadd_pause_level_iqr_db":"Pause variability",
    "qadd_speech_pause_level_contrast_db":"Speech–pause contrast",
    "qgain_typical_speech_level_dbfs":"Speech level",
    "qgain_within_segment_iqr_db":"Within-segment level var.",
    "qgain_between_segment_mad_db":"Between-segment level var.",
    "qgain_abs_drift_db_per_min":"Level drift",
    "qrev_srmr_norm":"SRMR",
    "qchan_ltas_distance_db":"LTAS distance",
    "qchan_rolloff95_deficit_hz":"Rolloff-95 deficit",
    "qchan_highband_ratio_deficit":"High-band deficit",
    "qchan_tilt_steepening_db_per_oct":"Tilt steepening",
}

# ------------------------------- Figure 3 -------------------------------
fig,axes=plt.subplots(2,2,figsize=(183*MM_TO_IN,135*MM_TO_IN),constrained_layout=True)

# a: paired diagnosis performance changes.
ax=axes[0,0]
d=authoritative_bootstrap.loc[authoritative_bootstrap["task"].eq("diagnosis") & authoritative_bootstrap["metric"].isin(["AUROC","Brier"])].copy()
d["label"]=d["contrast"].str.replace(" minus M_A","",regex=False)+" · Δ"+d["metric"]
y=np.arange(len(d)); x=d["estimate"].to_numpy(float); lo=d["ci_low"].to_numpy(float); hi=d["ci_high"].to_numpy(float)
colors=[PALETTE["M_A+Q"] if s.startswith("M_A+Q") else PALETTE["M_A-resQ"] for s in d["contrast"]]
for yi,xi,l,h,c in zip(y,x,lo,hi,colors):
    ax.errorbar(xi,yi,xerr=[[xi-l],[h-xi]],fmt="o",color=c,capsize=2,linewidth=1,markersize=4)
ax.axvline(0,color=PALETTE["neutral"],linestyle="--",linewidth=.7)
ax.set_yticks(y); ax.set_yticklabels(d["label"]); ax.invert_yaxis(); ax.set_xlabel("Paired performance change vs M_A"); clean_ax(ax); plabel(ax,"a")

# b: diagnosis absolute shift empirical survival curves.
ax=axes[0,1]
for comparison,color,label in [("M_A+Q minus M_A",PALETTE["M_A+Q"],"M_A+Q"),("M_A-resQ minus M_A",PALETTE["M_A-resQ"],"M_A-resQ")]:
    vals=np.sort(prediction_shifts.loc[prediction_shifts["task"].eq("diagnosis") & prediction_shifts["comparison"].eq(comparison),"absolute_shift"].to_numpy(float))
    if len(vals):
        surv=1-np.arange(1,len(vals)+1)/len(vals)
        ax.step(vals,surv,where="post",color=color,linewidth=1.4,label=label)
for anchor in [.05,.10]: ax.axvline(anchor,color=PALETTE["neutral"],linestyle=":" if anchor==.05 else "--",linewidth=.7)
ax.set_xlabel("Absolute participant probability shift"); ax.set_ylabel("Fraction with larger shift"); ax.set_ylim(0,1); ax.legend(frameon=False); clean_ax(ax); plabel(ax,"b")

# c: severity paired changes.
ax=axes[1,0]
s=authoritative_bootstrap.loc[authoritative_bootstrap["task"].eq("severity") & authoritative_bootstrap["metric"].isin(["MAE","RMSE"])].copy()
s["label"]=s["contrast"].str.replace(" minus M_A","",regex=False)+" · Δ"+s["metric"]
y=np.arange(len(s)); x=s["estimate"].to_numpy(float); lo=s["ci_low"].to_numpy(float); hi=s["ci_high"].to_numpy(float)
colors=[PALETTE["M_A+Q"] if z.startswith("M_A+Q") else PALETTE["M_A-resQ"] for z in s["contrast"]]
for yi,xi,l,h,c in zip(y,x,lo,hi,colors): ax.errorbar(xi,yi,xerr=[[xi-l],[h-xi]],fmt="o",color=c,capsize=2,linewidth=1,markersize=4)
ax.axvline(0,color=PALETTE["neutral"],linestyle="--",linewidth=.7); ax.set_yticks(y); ax.set_yticklabels(s["label"]); ax.invert_yaxis(); ax.set_xlabel("Paired error change vs M_A (points)"); clean_ax(ax); plabel(ax,"c")

# d: severity shifts.
ax=axes[1,1]
for comparison,color,label in [("M_A+Q minus M_A",PALETTE["M_A+Q"],"M_A+Q"),("M_A-resQ minus M_A",PALETTE["M_A-resQ"],"M_A-resQ")]:
    vals=np.sort(prediction_shifts.loc[prediction_shifts["task"].eq("severity") & prediction_shifts["comparison"].eq(comparison),"absolute_shift"].to_numpy(float))
    if len(vals):
        surv=1-np.arange(1,len(vals)+1)/len(vals); ax.step(vals,surv,where="post",color=color,linewidth=1.4,label=label)
ax.axvline(1.0,color=PALETTE["neutral"],linestyle="--",linewidth=.7)
ax.set_xlabel("Absolute predicted bulbar-score shift"); ax.set_ylabel("Fraction with larger shift"); ax.set_ylim(0,1); ax.legend(frameon=False); clean_ax(ax); plabel(ax,"d")

stem="Figure3_Goal2_inference_change_AUTHORITATIVE"
for ext,kwargs in [("pdf",{}),("svg",{}),("png",{"dpi":600})]: fig.savefig(FIGURES/f"{stem}.{ext}",bbox_inches="tight",**kwargs)
plt.show()

# ------------------------------- Figure 4 -------------------------------
primary_assoc=q_error_associations.loc[
    q_error_associations["cov_structure"].eq("independence")
    & q_error_associations["status"].eq("ok")
].copy()
q_order=CORE_Q
model_offsets={"M_A":-.18,"M_A+Q":0.0,"M_A-resQ":.18}


def make_figure4_for_task(task):
    local_assoc=primary_assoc.loc[primary_assoc["task"].eq(task)].copy()
    fig,axes=plt.subplots(2,2,figsize=(183*MM_TO_IN,175*MM_TO_IN),constrained_layout=True)

    def forest_models(ax,component):
        for model in MODELS:
            z=(local_assoc.loc[
                local_assoc["model"].eq(model) & local_assoc["component"].eq(component)
            ].set_index("q_feature").reindex(q_order))
            yy=np.arange(len(q_order))+model_offsets[model]
            x=z["effect_per_1IQR_Q"].to_numpy(float)
            lo=z["ci_low"].to_numpy(float); hi=z["ci_high"].to_numpy(float)
            ax.errorbar(x,yy,xerr=np.vstack([x-lo,hi-x]),fmt="o",color=PALETTE[model],
                        capsize=1.8,linewidth=.75,markersize=3,label=model)
        ax.axvline(0,color=PALETTE["neutral"],linestyle="--",linewidth=.7)
        ax.set_yticks(np.arange(len(q_order)))
        ax.set_yticklabels([Q_LABELS.get(q,q) for q in q_order])
        ax.invert_yaxis(); clean_ax(ax)

    forest_models(axes[0,0],"between")
    axes[0,0].set_xlabel("Δ OOF loss per 1-IQR between-person Q")
    axes[0,0].legend(frameon=False,ncol=3,loc="lower right")
    plabel(axes[0,0],"a")

    forest_models(axes[0,1],"within")
    axes[0,1].set_xlabel("Δ OOF loss per 1-IQR within-person Q")
    plabel(axes[0,1],"b")

    def forest_delta(ax,comparison_model,title,color):
        z=local_assoc.loc[local_assoc["model"].eq(comparison_model)].copy()
        for component,offset,marker in [("between",-.08,"o"),("within",.08,"s")]:
            zz=z.loc[z["component"].eq(component)].set_index("q_feature").reindex(q_order)
            yy=np.arange(len(q_order))+offset
            x=zz["effect_per_1IQR_Q"].to_numpy(float)
            lo=zz["ci_low"].to_numpy(float); hi=zz["ci_high"].to_numpy(float)
            ax.errorbar(x,yy,xerr=np.vstack([x-lo,hi-x]),fmt=marker,color=color,
                        capsize=1.8,linewidth=.75,markersize=3,label=component)
        ax.axvline(0,color=PALETTE["neutral"],linestyle="--",linewidth=.7)
        ax.set_yticks(np.arange(len(q_order)))
        ax.set_yticklabels([Q_LABELS.get(q,q) for q in q_order])
        ax.invert_yaxis(); ax.set_xlabel(title); ax.legend(frameon=False); clean_ax(ax)

    forest_delta(axes[1,0],"M_A+Q-minus-M_A-loss",
                 "Q dependence of paired loss change: M_A+Q − M_A",PALETTE["M_A+Q"])
    plabel(axes[1,0],"c")
    forest_delta(axes[1,1],"M_A-resQ-minus-M_A-loss",
                 "Q dependence of paired loss change: M_A-resQ − M_A",PALETTE["M_A-resQ"])
    plabel(axes[1,1],"d")

    stem=f"Figure4_Goal2_Q_dependent_error_{task}_AUTHORITATIVE"
    for ext,kwargs in [("pdf",{}),("svg",{}),("png",{"dpi":600})]:
        fig.savefig(FIGURES/f"{stem}.{ext}",bbox_inches="tight",**kwargs)
    plt.show()

make_figure4_for_task("diagnosis")
make_figure4_for_task("severity")

# -------------------------- Supplementary calibration --------------------------
fig,ax=plt.subplots(figsize=(89*MM_TO_IN,82*MM_TO_IN),constrained_layout=True)
ax.plot([0,1],[0,1],linestyle="--",color=PALETTE["neutral"],linewidth=.7)
cal_rows=[]
for model in MODELS:
    g=dx_final.loc[dx_final["model"].eq(model)].copy()
    smooth=lowess(g["y"].to_numpy(float),g["prediction"].to_numpy(float),frac=.40,it=0,return_sorted=True)
    ax.plot(smooth[:,0],np.clip(smooth[:,1],0,1),color=PALETTE[model],linewidth=1.2,label=model)
    intercept,slope=safe_calibration_stats(g["y"].to_numpy(float),g["prediction"].to_numpy(float))
    cal_rows.append({"model":model,"calibration_intercept_display":intercept,"calibration_slope_display":slope})
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_xlabel("Mean cross-fitted predicted probability"); ax.set_ylabel("Observed ALS fraction"); ax.legend(frameon=False); clean_ax(ax)
cal_display=pd.DataFrame(cal_rows); atomic_write_csv(cal_display,TABLES/"goal2_diagnosis_calibration_display_authoritative.csv")
stem="FigureS3_Goal2_native_calibration_AUTHORITATIVE"
for ext,kwargs in [("pdf",{}),("svg",{}),("png",{"dpi":600})]: fig.savefig(FIGURES/f"{stem}.{ext}",bbox_inches="tight",**kwargs)
plt.show()
print("AUTHORITATIVE FIGURES 3, 4 (diagnosis/severity), AND S3: WRITTEN")


## 28. Final Goal 2 completion seal

The seal requires the corrected protocol-concordant primary OOF, 2,000 paired bootstrap results, family ablations, Q-loss robustness, coefficient stability, and the HGB sensitivity. Optional Platt recalibration is explicitly not required because the frozen Methods label it optional and native calibration is the primary estimand. The overlap restriction is conditional; diagnostics are saved and must be reviewed before deciding whether that conditional sensitivity is triggered.


In [ ]:

required_files=[
    OOF_DIR/"goal2_diagnosis_oof_authoritative.csv",
    OOF_DIR/"goal2_severity_oof_authoritative.csv",
    TABLES/"goal2_protocol_concordance_audit.csv",
    TABLES/"goal2_corrected_residualizer_fold_manifest.csv",
    TABLES/"goal2_authoritative_observed_metrics.csv",
    TABLES/"goal2_metrics_authoritative.csv",
    TABLES/"goal2_prediction_shifts_authoritative.csv",
    TABLES/"goal2_q_error_associations.csv",
    TABLES/"goal2_q_loss_sensitivity.csv",
    TABLES/"goal2_sensitivity_summary.csv",
    TABLES/"goal2_family_ablations.csv",
    TABLES/"goal2_ridge_coefficient_stability.csv",
    TABLES/"goal2_overlap_diagnostics.csv",
    FIGURES/"FigureS6_Goal2_completion_sensitivities.pdf",
    FIGURES/"Figure3_Goal2_inference_change_AUTHORITATIVE.pdf",
    FIGURES/"Figure4_Goal2_Q_dependent_error_diagnosis_AUTHORITATIVE.pdf",
    FIGURES/"Figure4_Goal2_Q_dependent_error_severity_AUTHORITATIVE.pdf",
    FIGURES/"FigureS3_Goal2_native_calibration_AUTHORITATIVE.pdf",
]
if RUN_HGB:
    required_files.append(TABLES/"goal2_hgb_fold_manifest.csv")
bad=[str(p) for p in required_files if (not p.exists()) or p.stat().st_size==0]
if bad:
    raise RuntimeError("Goal 2 completion cannot be sealed; missing/empty artifacts:\n"+"\n".join(bad))
if not RUN_HGB:
    raise RuntimeError("Goal 2 complete seal requires RUN_HGB=True.")

# Hash every final completion artifact.
artifact_hashes={str(p.relative_to(ROOT)):sha256_file(p) for p in required_files}
completion_manifest={
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "status":"PASS_PENDING_OVERLAP_REVIEW",
    "engine_version":"goal2-completion-v1.0.0",
    "primary_notebook20_success":primary_success,
    "a_freeze_version":a_manifest["a_freeze_version"],
    "paper1_commit":observed_paper1_commit,
    "outer_cv":"5 folds x 10 repeats, participant grouped",
    "inner_cv":"5 participant-grouped folds",
    "bootstrap_replicates":N_COMPLETION_BOOTSTRAPS,
    "residualizer_alpha_rule":"outer-training grouped CV on standardized A reconstruction error over 1e-4...1e4",
    "optional_platt":"not run; optional by frozen Methods, native calibration remains primary",
    "overlap_restriction":"conditional; inspect goal2_overlap_diagnostics.csv before final DONE seal",
    "artifact_hashes":artifact_hashes,
    "interpretation_boundary":"Goal 2 demonstrates observational model dependence/sensitivity, not a unique physical technical cause.",
}
atomic_write_json(completion_manifest,OUT/"GOAL2_COMPLETION_MANIFEST.json")

print("\nGOAL 2 COMPLETION ANALYSES: PASS")
print("Status: PASS_PENDING_OVERLAP_REVIEW")
print("Next action: send me the final tables printed below, especially overlap diagnostics, so we can decide whether the conditional overlap-restricted sensitivity is triggered and then write DONE.json.")
print("Output directory:",OUT)
display(authoritative_bootstrap)
display(family_ablations)
display(overlap_diagnostics)
